# Prescriptively Unsafe: A Reproducible Audit of When Accurate ICU Prediction Models Would Harm Patients If Used to Guide Treatment

## Notebook 3 of 3: Publication assembly, reviewer hardening, and manuscript support

**Purpose.** Notebooks 1 and 2 built the analytic cohorts and the Prescriptive Safety Audit. This notebook turns that work into an elite-journal submission. It does two things. First, it assembles the publication package: the numbered main tables, the journal-ready figure set, the supplementary material, and the manuscript-support text, all populated from the saved outputs of the earlier notebooks. Second, and more important, it adds the inferential analyses a strong reviewer will demand and that were not yet present, so that the central claim survives adversarial reading.

**What this notebook adds beyond Notebook 2.** Each of these was chosen because a specific reviewer would otherwise reject the paper for its absence.

- **Randomisation (permutation) inference.** A p-value for the headline result by testing the observed deployment hazard against the null in which predicted risk and estimated benefit are unrelated. Without this, prescriptive harm is a point estimate with no significance.
- **Negative controls.** A negative-control outcome that treatment cannot cause, to show the causal pipeline returns a null when it should, and a sham-treatment policy to show the audit does not manufacture harm from a random action. These are the standard falsification tests for observational causal work.
- **Treatment-effect model validation.** The c-for-benefit and benefit-decile calibration, because the entire audit rests on the conditional treatment effect and a reviewer will ask whether that effect model discriminates and calibrates at all.
- **Uncertainty propagation.** Bootstrap intervals that carry the treatment-effect estimation error into the hazard metrics, not just patient sampling error.
- **A tipping-point analysis.** How large a systematic bias in the treatment-effect estimate would have to be to overturn the safety conclusion, the prescriptive analogue of the E-value.
- **A structured reviewer stress test.** An explicit criticism-to-response matrix across the six reviewer personas, each row linked to the analysis that answers it.

**Scientific honesty.** We keep the vocabulary disciplined throughout. Prediction and association are distinguished from counterfactual estimation and causal inference, and both are distinguished from the deployment simulation that defines the audit. Every causal statement is conditional on the stated identifying assumptions, and the limitations section states plainly what public observational ICU data cannot establish.

**Conventions.** British spelling, no em dashes, exact numerical anchoring, audit in preference to framework. Outputs are organised under `nb3_outputs/` into `tables/`, `figures/`, `supplementary/`, and `manuscript/`. Figures are written at 300 dpi as PNG and vector PDF.

In [1]:
# =============================================================================
# CONFIGURATION
# =============================================================================
from pathlib import Path

class CFG:
    # inputs from the earlier notebooks
    NB1_DIR = Path("./nb1_outputs")
    NB2_DIR = Path("./nb2_outputs")
    MIMIC_PARQUET = NB1_DIR / "mimic_analytic.parquet"
    EICU_PARQUET  = NB1_DIR / "eicu_analytic.parquet"
    NB1_MANIFEST  = NB1_DIR / "nb1_manifest.json"
    NB2_MANIFEST  = NB2_DIR / "nb2_manifest.json"
    NB2_ARRAYS    = NB2_DIR / "nb2_audit_arrays.npz"
    NB2_TABLES    = NB2_DIR / "tables"

    # publication output tree
    OUT       = Path("./nb3_outputs")
    TAB_DIR   = OUT / "tables"
    FIG_DIR   = OUT / "figures"
    SUP_DIR   = OUT / "supplementary"
    MAN_DIR   = OUT / "manuscript"

    # audit parameters (must match Notebook 2)
    OUTCOME       = "in_hospital_mortality"
    TREATMENT     = "any_organ_support_post"
    BENEFIT_DELTA = 0.01

    RANDOM_SEED   = 20260707
    N_PERMUTE     = 2000     # permutation-test replicates
    N_BOOTSTRAP   = 500      # bootstrap replicates for intervals
    CATE_BOOT     = 40       # treatment-effect model bootstrap draws
    FAST          = True
    FIG_DPI       = 300

for d in (CFG.OUT, CFG.TAB_DIR, CFG.FIG_DIR, CFG.SUP_DIR, CFG.MAN_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("Notebook 3 outputs ->", CFG.OUT.resolve())


Notebook 3 outputs -> C:\Users\<user>\Downloads\nb3_outputs


In [2]:
# =============================================================================
# ENVIRONMENT, LOGGING, REPRODUCIBILITY
# =============================================================================
import os, sys, json, time, logging, platform, warnings, random, textwrap
from contextlib import contextmanager
from datetime import datetime
import numpy as np, pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch, Rectangle, Circle
from matplotlib.lines import Line2D

try:
    from scipy import stats as sstats
    _HAS_SCIPY = True
except Exception:
    _HAS_SCIPY = False

warnings.filterwarnings("ignore")
random.seed(CFG.RANDOM_SEED); np.random.seed(CFG.RANDOM_SEED)

logger = logging.getLogger("psa3"); logger.setLevel(logging.INFO); logger.handlers.clear()
_fmt = logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S")
_sh = logging.StreamHandler(sys.stdout); _sh.setFormatter(_fmt); logger.addHandler(_sh)
_fh = logging.FileHandler(CFG.OUT / "nb3_run.log", mode="w", encoding="utf-8"); _fh.setFormatter(_fmt)
logger.addHandler(_fh); logger.propagate = False

@contextmanager
def timed(label):
    t = time.time(); logger.info("START  %s", label)
    try: yield
    finally: logger.info("DONE   %s  (%.1fs)", label, time.time() - t)

def savefig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(CFG.FIG_DIR / f"{name}.{ext}", dpi=CFG.FIG_DPI, bbox_inches="tight")
    plt.close(fig); logger.info("figure -> %s", name)

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": CFG.FIG_DPI, "axes.grid": True,
                     "grid.alpha": 0.25, "font.size": 10, "axes.spines.top": False,
                     "axes.spines.right": False})
PALETTE = {"MIMIC": "#3b76af", "eICU": "#5aa469", "harm": "#c0392b", "safe": "#2e8b57",
           "neutral": "#7f8c8d", "accent": "#8e44ad", "gold": "#d4a017"}
logger.info("Python %s | numpy %s | pandas %s | scipy=%s",
            sys.version.split()[0], np.__version__, pd.__version__, _HAS_SCIPY)


C:\Users\<user>\anaconda3-v2\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


00:55:03 | INFO    | Python 3.11.5 | numpy 1.24.4 | pandas 2.3.3 | scipy=True


In [3]:
# =============================================================================
# LOAD ALL REUSABLE OUTPUTS FROM NOTEBOOKS 1 AND 2 (no retraining)
# =============================================================================
def load_json(p, default=None):
    return json.load(open(p)) if Path(p).exists() else (default if default is not None else {})

nb1_manifest = load_json(CFG.NB1_MANIFEST)
nb2_manifest = load_json(CFG.NB2_MANIFEST)

# per-patient audit vectors cached by Notebook 2
arr = np.load(CFG.NB2_ARRAYS)
THR         = arr["thresholds"]
r_hold, r_ext         = arr["r_hold"], arr["r_ext"]
benefit_hold, benefit_ext = arr["benefit_hold"], arr["benefit_ext"]
a_hold, a_ext         = arr["a_hold"], arr["a_ext"]
y_hold, y_ext         = arr["y_hold"], arr["y_ext"]
logger.info("Loaded audit vectors: hold n=%d, ext n=%d", len(r_hold), len(r_ext))

# Notebook 2 result tables (whichever exist)
def load_table(name):
    p = CFG.NB2_TABLES / name
    return pd.read_csv(p) if p.exists() else None
T = {name.split(".")[0]: load_table(name) for name in [
        "model_performance.csv", "delong_tests.csv", "ate_estimates.csv",
        "prescriptive_safety_scores.csv", "headline_ci.csv", "robustness_battery.csv",
        "subgroup_prescriptive_safety.csv", "ablation_mitigation.csv",
        "covariate_balance.csv", "importance_risk_model.csv"]}
evalue   = load_json(CFG.NB2_TABLES / "evalue.json", {})
tii_sum  = load_json(CFG.NB2_TABLES / "tii_summary.json", {})
overlap  = load_json(CFG.NB2_TABLES / "risk_treatment_overlap.json", {})
positivity = load_json(CFG.NB2_TABLES / "positivity_overlap.json", {})
PRIMARY  = nb2_manifest.get("primary_model", "boosting")
logger.info("Reused %d Notebook 2 tables | primary model = %s",
            sum(v is not None for v in T.values()), PRIMARY)

# analytic tables (read-only reuse for Table 1 and negative controls)
mimic = pd.read_parquet(CFG.MIMIC_PARQUET)
eicu  = pd.read_parquet(CFG.EICU_PARQUET)
logger.info("Analytic tables: MIMIC %d, eICU %d", len(mimic), len(eicu))


00:55:03 | INFO    | Loaded audit vectors: hold n=10821, ext n=86403
00:55:03 | INFO    | Reused 10 Notebook 2 tables | primary model = lightgbm
00:55:03 | INFO    | Analytic tables: MIMIC 36069, eICU 86403


In [4]:
# =============================================================================
# REBUILD THE AUDIT PRIMITIVES (identical definitions to Notebook 2)
# so this notebook is self-contained from the cached vectors.
# =============================================================================
DELTA = CFG.BENEFIT_DELTA
def deployment_delta(r, benefit, a, tau):
    action = (r >= tau).astype(int)
    withdraw = (a == 1) & (action == 0)
    add      = (a == 0) & (action == 1)
    delta = np.where(withdraw, benefit, np.where(add, -benefit, 0.0))
    return delta, (action != a)
def metrics_at(r, benefit, a, tau):
    delta, flip = deployment_delta(r, benefit, a, tau)
    harmful = flip & (delta > DELTA)
    n = len(r)
    hrr = float(harmful.mean())
    hfr = float(harmful.sum() / flip.sum()) if flip.sum() > 0 else np.nan
    eed_net = float(delta.sum() / n * 1000)
    eed_harm = float(delta[harmful].sum() / n * 1000)
    car = float(np.mean((r >= tau) == (benefit > DELTA)))
    return hrr, hfr, eed_net, eed_harm, car
def sweep(r, benefit, a, thresholds=THR):
    rows = [metrics_at(r, benefit, a, t) for t in thresholds]
    keys = ["hrr", "hfr", "eed_net", "eed_harm", "car"]
    out = {k: np.array([row[i] for row in rows]) for i, k in enumerate(keys)}
    n = len(r); eed_wd, eed_add = [], []
    for t in thresholds:
        delta, _ = deployment_delta(r, benefit, a, t)
        action = (r >= t).astype(int)
        withdraw = (a == 1) & (action == 0); add = (a == 0) & (action == 1)
        eed_wd.append(float(delta[withdraw].sum() / n * 1000))
        eed_add.append(float(delta[add].sum() / n * 1000))
    out["eed_wd"] = np.array(eed_wd); out["eed_add"] = np.array(eed_add)
    return out
def scalars(sw):
    eed = sw["eed_net"]
    return {"PSS": float(np.nanmean(1 - sw["hfr"])),
            "DHI": float(np.mean(eed)),                          # canonical: SIGNED net EED/1000
            "DHI_onesided": float(np.mean(np.maximum(eed, 0))),  # reference only, NOT headline
            "DHI_signed": float(np.mean(eed)),
            "DHI_frac_positive": float(np.mean(eed > 0)),
            "DHI_withdraw": float(np.mean(sw["eed_wd"])),        # signed, not max(,0)
            "DHI_add": float(np.mean(sw["eed_add"])),           # signed, not max(,0)
            "DHI_frac": float(np.mean(eed > 0)),
            "CAR_mean": float(np.nanmean(sw["car"]))}
def rank_concordance(r, benefit, n_pairs=40000, seed=CFG.RANDOM_SEED):
    rng = np.random.default_rng(seed); n = len(r)
    i, j = rng.integers(0, n, n_pairs), rng.integers(0, n, n_pairs)
    ok = (r[i] != r[j]) & (benefit[i] != benefit[j])
    return float(((r[i] > r[j]) == (benefit[i] > benefit[j]))[ok].mean())
sw_hold = sweep(r_hold, benefit_hold, a_hold)
sw_ext  = sweep(r_ext,  benefit_ext,  a_ext)
sc_hold, sc_ext = scalars(sw_hold), scalars(sw_ext)
tau_ref = float(y_hold.mean())
logger.info("Recovered headline: PSS hold %.3f / ext %.3f | DHI hold %.2f / ext %.2f (add %.2f / withdraw %.2f)",
            sc_hold["PSS"], sc_ext["PSS"], sc_hold["DHI"], sc_ext["DHI"], sc_hold["DHI_add"], sc_hold["DHI_withdraw"])
print("Audit primitives reconstructed. Reference threshold tau_ref =", round(tau_ref, 3))

00:55:03 | INFO    | Recovered headline: PSS hold 0.821 / ext 0.856 | DHI hold -10.16 / ext -12.16 (add 5.09 / withdraw -15.25)
Audit primitives reconstructed. Reference threshold tau_ref = 0.106


In [5]:
SEEDS = [20260707, 11, 202, 3037, 99991]
dhi_seeds = []
for s in SEEDS:
    rng = np.random.default_rng(s)
    idx = rng.integers(0, len(r_hold), len(r_hold))
    dhi_seeds.append(scalars(sweep(r_hold[idx], benefit_hold[idx], a_hold[idx]))["DHI"])
logger.info("DHI stability across %d seeds: %.2f +/- %.2f", len(SEEDS), np.mean(dhi_seeds), np.std(dhi_seeds))
json.dump({"seeds": SEEDS, "DHI_mean": float(np.mean(dhi_seeds)), "DHI_sd": float(np.std(dhi_seeds))},
          open(CFG.TAB_DIR / "seed_stability.json", "w"), indent=2)

00:55:03 | INFO    | DHI stability across 5 seeds: -9.98 +/- 0.37


---
## A. Study design and conceptual figures

Elite reviewers read the figures first. This section produces the schematic backbone of the paper: the causal diagram that states our identification argument, the landmark study-design timeline, the audit-overview schematic, the cohort flow, and a graphical abstract that names the paper's one idea, that prediction accuracy and prescriptive safety are different axes.

In [6]:
# =============================================================================
# Figure: causal directed acyclic graph
# =============================================================================
fig, ax = plt.subplots(figsize=(9, 5.4)); ax.axis("off"); ax.grid(False)
nodes = {"L": (0.15, 0.55, "L\npre-anchor\ncovariates"),
         "A": (0.5, 0.8, "A\npost-anchor\ntreatment"),
         "Y": (0.85, 0.55, "Y\nin-hospital\nmortality"),
         "U": (0.5, 0.18, "U\nunmeasured\nconfounding"),
         "r": (0.15, 0.15, "r = f(L)\nrisk score")}
for k, (x, y, lab) in nodes.items():
    col = PALETTE["accent"] if k == "A" else (PALETTE["harm"] if k == "Y" else
          (PALETTE["neutral"] if k == "U" else PALETTE["MIMIC"]))
    style = "dashed" if k == "U" else "solid"
    b = FancyBboxPatch((x - 0.09, y - 0.07), 0.18, 0.14, boxstyle="round,pad=0.02",
                       fc="white", ec=col, lw=2, ls=style)
    ax.add_patch(b); ax.text(x, y, lab, ha="center", va="center", fontsize=9)
def arrow(a, b, ls="solid", col="k"):
    (x0, y0, _), (x1, y1, _) = nodes[a], nodes[b]
    ax.add_patch(FancyArrowPatch((x0, y0), (x1, y1), arrowstyle="-|>", mutation_scale=16,
                 lw=1.6, color=col, ls=ls, shrinkA=22, shrinkB=22,
                 connectionstyle="arc3,rad=0.0"))
arrow("L", "A"); arrow("L", "Y"); arrow("A", "Y", col=PALETTE["accent"])
arrow("U", "A", ls="dashed", col=PALETTE["neutral"]); arrow("U", "Y", ls="dashed", col=PALETTE["neutral"])
arrow("L", "r", col=PALETTE["MIMIC"])
ax.annotate("deployment: r guides A\n(the path the audit perturbs)", xy=(0.32, 0.8), xytext=(0.02, 0.93),
            fontsize=8, color=PALETTE["accent"],
            arrowprops=dict(arrowstyle="->", color=PALETTE["accent"], lw=1.2))
ax.text(0.5, 0.02, "Solid: assumed causal structure under exchangeability given L.  "
        "Dashed: unmeasured confounding probed by the E-value.",
        ha="center", fontsize=7.5, color=PALETTE["neutral"])
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_title("Causal DAG: treatment lies on the path from covariates to outcome", fontsize=11)
savefig(fig, "fig_causal_dag")
print("Causal DAG written.")


00:55:04 | INFO    | figure -> fig_causal_dag
Causal DAG written.


In [7]:
# =============================================================================
# Figure: landmark study-design timeline and audit-overview schematic
# =============================================================================
fig, axes = plt.subplots(2, 1, figsize=(11, 7)); [a.axis("off") or a.grid(False) for a in axes]
# --- timeline ---
ax = axes[0]
ax.plot([0.05, 0.95], [0.5, 0.5], color="k", lw=2)
for x, lab, sub in [(0.15, "ICU admission", "t = 0"),
                    (0.45, "decision anchor", "t0 + 24 h"),
                    (0.9, "discharge / death", "outcome Y")]:
    ax.plot([x], [0.5], "o", color=PALETTE["MIMIC"], ms=10)
    ax.text(x, 0.62, lab, ha="center", fontsize=9)
    ax.text(x, 0.4, sub, ha="center", fontsize=8, color=PALETTE["neutral"])
ax.add_patch(Rectangle((0.15, 0.68), 0.30, 0.12, fc=PALETTE["MIMIC"], alpha=0.2))
ax.text(0.30, 0.74, "covariates L (measured)", ha="center", fontsize=8)
ax.add_patch(Rectangle((0.45, 0.2), 0.45, 0.12, fc=PALETTE["accent"], alpha=0.2))
ax.text(0.675, 0.26, "treatment A (organ support)", ha="center", fontsize=8)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_title("Landmark design: covariates strictly precede the anchor, treatment strictly follows", fontsize=10)
# --- audit overview ---
ax = axes[1]
boxes = [(0.1, "Prediction\nr = f(L)", PALETTE["MIMIC"]),
         (0.32, "Causal effect\nb = -CATE", PALETTE["accent"]),
         (0.54, "Deployment rule\nwithhold if r<tau", PALETTE["gold"]),
         (0.76, "Harm Delta and\nPSS, DHI, HRR", PALETTE["harm"])]
for x, lab, col in boxes:
    b = FancyBboxPatch((x - 0.09, 0.4), 0.18, 0.3, boxstyle="round,pad=0.02",
                       fc="white", ec=col, lw=2); ax.add_patch(b)
    ax.text(x, 0.55, lab, ha="center", va="center", fontsize=8.5)
for x0, x1 in [(0.19, 0.23), (0.41, 0.45), (0.63, 0.67)]:
    ax.add_patch(FancyArrowPatch((x0, 0.55), (x1, 0.55), arrowstyle="-|>", mutation_scale=14, lw=1.5, color="k"))
ax.set_xlim(0, 1); ax.set_ylim(0.2, 0.9)
ax.set_title("Prescriptive Safety Audit: from prediction and effect to deployment harm", fontsize=10)
savefig(fig, "fig_study_design")
print("Study-design and audit-overview schematic written.")


00:55:05 | INFO    | figure -> fig_study_design
Study-design and audit-overview schematic written.


In [8]:
# =============================================================================
# Figure: CONSORT-style cohort flow (from manifests; robust to missing detail)
# =============================================================================
n_mimic = int(nb1_manifest.get("cohorts", {}).get("mimic_stays", len(mimic)))
n_eicu  = int(nb1_manifest.get("cohorts", {}).get("eicu_stays", len(eicu)))
fig, ax = plt.subplots(figsize=(9, 6)); ax.axis("off"); ax.grid(False)
def box(x, y, w, h, text, col=PALETTE["MIMIC"]):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.01", fc="white", ec=col, lw=1.8))
    ax.text(x + w/2, y + h/2, text, ha="center", va="center", fontsize=8.5)
box(0.30, 0.85, 0.40, 0.10, f"MIMIC-IV source ICU stays\n(development and internal test)")
box(0.30, 0.68, 0.40, 0.10, f"Landmark-eligible at 24 h\nMIMIC n = {n_mimic}", PALETTE["MIMIC"])
box(0.05, 0.45, 0.40, 0.12, f"MIMIC development + internal\nhold-out (70/30 split)\nn = {n_mimic}", PALETTE["MIMIC"])
box(0.55, 0.45, 0.40, 0.12, f"eICU-CRD external validation\nn = {n_eicu}", PALETTE["eICU"])
box(0.30, 0.22, 0.40, 0.12, f"Prescriptive Safety Audit\noutcome: {CFG.OUTCOME}\ntreatment: {CFG.TREATMENT}", PALETTE["harm"])
for x0, y0, x1, y1 in [(0.5, 0.85, 0.5, 0.78), (0.5, 0.68, 0.5, 0.585),
                       (0.5, 0.585, 0.25, 0.57), (0.5, 0.585, 0.75, 0.57),
                       (0.25, 0.45, 0.45, 0.34), (0.75, 0.45, 0.55, 0.34)]:
    ax.add_patch(FancyArrowPatch((x0, y0), (x1, y1), arrowstyle="-|>", mutation_scale=13, lw=1.4, color="k"))
ax.set_xlim(0, 1); ax.set_ylim(0.15, 1)
ax.set_title("Cohort derivation and analysis flow", fontsize=11)
savefig(fig, "fig_consort_flow")
print(f"CONSORT flow written (MIMIC {n_mimic}, eICU {n_eicu}).")


00:55:06 | INFO    | figure -> fig_consort_flow
CONSORT flow written (MIMIC 36069, eICU 86403).


In [9]:
# =============================================================================
# Figure: graphical abstract - accuracy and prescriptive safety are different axes
# =============================================================================
fig, ax = plt.subplots(figsize=(8.5, 7))
auroc_hold = float(nb2_manifest.get("tii", {}).get("AUROC_blind", 0.75))
models_scores = pd.DataFrame(nb2_manifest.get("prescriptive_scores", []))
ax.axhline(0.5, color=PALETTE["neutral"], lw=0.8, ls=":")
ax.axvspan(0, 0.5, color=PALETTE["harm"], alpha=0.06)
ax.text(0.25, 0.05, "prescriptively unsafe region", color=PALETTE["harm"], fontsize=9, ha="center")
if not models_scores.empty:
    hold = models_scores[models_scores["cohort"] == "MIMIC-hold"]
    for _, row in hold.iterrows():
        ax.scatter(row["PSS"], 0.5 + (row["CAR_mean"] - 0.5), s=90, color=PALETTE["MIMIC"], zorder=3)
        ax.annotate(row["model"], (row["PSS"], 0.5 + (row["CAR_mean"] - 0.5)),
                    fontsize=7, xytext=(4, 4), textcoords="offset points")
ax.scatter([sc_hold["PSS"]], [0.5 + (sc_hold["CAR_mean"] - 0.5)], s=220, marker="*",
           color=PALETTE["gold"], edgecolor="k", zorder=4, label=f"primary ({PRIMARY})")
ax.set_xlabel("Prescriptive Safety Score (higher = safer)")
ax.set_ylabel("benefit-risk alignment (CAR)")
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_title("Accurate models occupy the whole safety range:\nprediction quality does not determine prescriptive safety", fontsize=11)
ax.legend(loc="upper left", fontsize=8)
ax.text(0.98, 0.02, f"internal AUROC approx {auroc_hold:.2f}", ha="right", fontsize=8, color=PALETTE["neutral"])
savefig(fig, "fig_graphical_abstract")
print("Graphical abstract written.")


00:55:06 | INFO    | figure -> fig_graphical_abstract
Graphical abstract written.


---
## B. Publication-ready tables

We assemble the numbered main tables from the saved outputs, computing only what was not already stored. To attach patient characteristics and subgroup labels to the cached predictions without retraining, we first reproduce Notebook 2's internal hold-out split deterministically, using the identical seed and stratification, and verify the reconstruction against the cached outcome and treatment vectors. This is both an enabling step and an integrity check.

In [10]:
# =============================================================================
# Reconstruct and verify the Notebook 2 internal hold-out split
# =============================================================================
from sklearn.model_selection import train_test_split
_, mimic_hold = train_test_split(mimic, test_size=0.30, random_state=CFG.RANDOM_SEED,
                                 stratify=mimic[CFG.OUTCOME])
mimic_hold = mimic_hold.reset_index(drop=True)

HAVE_HOLD_META = (len(mimic_hold) == len(y_hold)
                  and np.array_equal(mimic_hold[CFG.OUTCOME].astype(int).values, y_hold.astype(int))
                  and np.array_equal(mimic_hold[CFG.TREATMENT].astype(int).values, a_hold.astype(int)))
if HAVE_HOLD_META:
    logger.info("Hold-out split reconstructed and verified against cached vectors (n=%d).", len(mimic_hold))
    hold_meta = mimic_hold.copy()
    hold_meta["age_band"] = pd.cut(hold_meta["admission_age"], [0, 50, 65, 80, 200],
                                   labels=["<50", "50-64", "65-79", "80+"])
else:
    logger.warning("Could not verify hold-out reconstruction; subgroup CIs will reuse Notebook 2 point estimates.")
    hold_meta = None
print("Hold-out metadata aligned:", HAVE_HOLD_META)


00:55:07 | INFO    | Hold-out split reconstructed and verified against cached vectors (n=10821).
Hold-out metadata aligned: True


In [11]:
# =============================================================================
# TABLE 1. Baseline cohort characteristics with standardised mean differences
# =============================================================================
def fmt_cont(s):
    s = pd.to_numeric(s, errors="coerce").dropna()
    return f"{s.mean():.1f} ({s.std():.1f})"
def fmt_bin(s):
    s = pd.to_numeric(s, errors="coerce"); return f"{int(s.sum())} ({100*s.mean():.1f}%)"
def smd_cont(a, b):
    a = pd.to_numeric(a, errors="coerce").dropna(); b = pd.to_numeric(b, errors="coerce").dropna()
    return (a.mean() - b.mean()) / np.sqrt(0.5 * (a.var() + b.var()) + 1e-12)
def smd_bin(a, b):
    pa, pb = np.nanmean(a), np.nanmean(b)
    return (pa - pb) / np.sqrt(0.5 * (pa*(1-pa) + pb*(1-pb)) + 1e-12)

cont_vars = [("Age, years", "admission_age"), ("SOFA total", "sofa_total"),
             ("Lactate max, mmol/L", "lactate_max"), ("Creatinine max, mg/dL", "creatinine_max"),
             ("MAP min, mmHg", "mbp_min"), ("Elixhauser count", "elix_count")]
bin_vars  = [("In-hospital mortality", "in_hospital_mortality"),
             ("Organ support (post-anchor)", "any_organ_support_post"),
             ("Vasopressors", "vaso_post"), ("Mechanical ventilation", "vent_post"),
             ("Renal replacement", "rrt_post")]
rows = []
for lab, col in cont_vars:
    if col in mimic.columns:
        rows.append({"Characteristic": lab, "MIMIC-IV": fmt_cont(mimic[col]),
                     "eICU-CRD": fmt_cont(eicu[col]),
                     "SMD": round(smd_cont(mimic[col], eicu[col]), 3)})
for lab, col in bin_vars:
    if col in mimic.columns:
        rows.append({"Characteristic": lab, "MIMIC-IV": fmt_bin(mimic[col]),
                     "eICU-CRD": fmt_bin(eicu[col]),
                     "SMD": round(smd_bin(mimic[col], eicu[col]), 3)})
# sex distribution
if "sex" in mimic.columns:
    for lvl in sorted(set(mimic["sex"].dropna().unique())):
        rows.append({"Characteristic": f"Sex = {lvl}",
                     "MIMIC-IV": fmt_bin((mimic["sex"] == lvl).astype(int)),
                     "eICU-CRD": fmt_bin((eicu["sex"] == lvl).astype(int)),
                     "SMD": round(smd_bin((mimic["sex"] == lvl).astype(int),
                                          (eicu["sex"] == lvl).astype(int)), 3)})
table1 = pd.DataFrame(rows)
table1.insert(1, "Overall n", [f"MIMIC {len(mimic)} / eICU {len(eicu)}"] + [""] * (len(table1) - 1))
table1.to_csv(CFG.TAB_DIR / "table1_baseline_characteristics.csv", index=False)
with open(CFG.TAB_DIR / "table1_baseline_characteristics.tex", "w") as f:
    f.write(table1.to_latex(index=False, escape=True,
            caption="Baseline characteristics of the MIMIC-IV and eICU-CRD landmark cohorts.",
            label="tab:baseline"))
logger.info("Table 1 written (%d rows).", len(table1))
table1


00:55:07 | INFO    | Table 1 written (13 rows).


,Characteristic,Overall n,MIMIC-IV,eICU-CRD,SMD
0,"Age, years",MIMIC 36069 / eICU 86403,65.0 (16.8),63.9 (16.8),0.062
1,SOFA total,,6.4 (4.4),4.1 (3.3),0.596
2,"Lactate max, mmol/L",,2.8 (2.2),2.8 (2.8),-0.005
3,"Creatinine max, mg/dL",,1.4 (1.5),1.6 (1.7),-0.088
4,"MAP min, mmHg",,59.5 (12.6),57.5 (15.5),0.141
5,Elixhauser count,,4.3 (2.3),1.1 (1.4),1.668
6,In-hospital mortality,,3826 (10.6%),8568 (9.9%),0.023
7,Organ support (post-anchor),,13485 (37.4%),39456 (45.7%),-0.169
8,Vasopressors,,8236 (22.8%),14446 (16.7%),0.154
9,Mechanical ventilation,,10486 (29.1%),33700 (39.0%),-0.211


In [12]:
# =============================================================================
# TABLE 2. Model performance (discrimination, calibration) with DeLong tests
# =============================================================================
if T["model_performance"] is not None:
    mp = T["model_performance"].copy()
    mp["AUROC (95% CI)"] = mp.apply(lambda r: f"{r['AUROC']:.3f} ({r['AUROC_lo']:.3f}-{r['AUROC_hi']:.3f})", axis=1)
    table2 = mp[["model", "cohort", "AUROC (95% CI)", "AUPRC", "Brier", "ECE", "MCC"]].round(3)
    table2.to_csv(CFG.TAB_DIR / "table2_model_performance.csv", index=False)
    with open(CFG.TAB_DIR / "table2_model_performance.tex", "w") as f:
        f.write(table2.to_latex(index=False, escape=True,
                caption="Discrimination and calibration on the internal MIMIC-IV hold-out and external eICU-CRD cohorts.",
                label="tab:performance"))
    logger.info("Table 2 written.")
else:
    table2 = pd.DataFrame(); logger.warning("model_performance table missing.")
if T["delong_tests"] is not None:
    T["delong_tests"].to_csv(CFG.TAB_DIR / "table2b_delong_tests.csv", index=False)
table2


00:55:07 | INFO    | Table 2 written.


,model,cohort,AUROC (95% CI),AUPRC,Brier,ECE,MCC
0,logreg,MIMIC-hold,0.893 (0.884-0.902),0.526,0.068,0.008,0.387
1,logreg,eICU-ext,0.813 (0.809-0.818),0.368,0.077,0.030,0.286
2,random_forest,MIMIC-hold,0.891 (0.882-0.899),0.523,0.068,0.007,0.390
3,random_forest,eICU-ext,0.809 (0.804-0.813),0.354,0.080,0.033,0.291
4,hist_gb,MIMIC-hold,0.905 (0.898-0.913),0.557,0.065,0.008,0.425
5,hist_gb,eICU-ext,0.816 (0.811-0.820),0.367,0.079,0.040,0.292
6,lightgbm,MIMIC-hold,0.909 (0.901-0.916),0.569,0.064,0.008,0.438
7,lightgbm,eICU-ext,0.822 (0.817-0.826),0.381,0.078,0.037,0.302
8,xgboost,MIMIC-hold,0.907 (0.899-0.915),0.561,0.064,0.006,0.431
9,xgboost,eICU-ext,0.821 (0.816-0.825),0.378,0.078,0.038,0.296


In [13]:
# =============================================================================
# TABLE 3. Prescriptive safety audit results (all models, both cohorts)
# =============================================================================
if T["prescriptive_safety_scores"] is not None:
    ps = T["prescriptive_safety_scores"].copy()
    table3 = ps[["model", "cohort", "PSS", "DHI", "DHI_frac", "CAR_mean", "concordance_r_b"]].round(3)
    # attach bootstrap CIs for the primary model from Notebook 2 headline_ci
    if T["headline_ci"] is not None:
        hc = T["headline_ci"]
        note = {(r["cohort"], r["metric"]): f"[{r['CI_lo']:.3f}, {r['CI_hi']:.3f}]"
                for _, r in hc.iterrows()}
        table3["PSS 95% CI"] = table3.apply(
            lambda r: note.get((r["cohort"], "PSS"), "") if r["model"] == PRIMARY else "", axis=1)
        table3["DHI 95% CI"] = table3.apply(
            lambda r: note.get((r["cohort"], "DHI"), "") if r["model"] == PRIMARY else "", axis=1)
    table3.to_csv(CFG.TAB_DIR / "table3_prescriptive_safety.csv", index=False)
    with open(CFG.TAB_DIR / "table3_prescriptive_safety.tex", "w") as f:
        f.write(table3.to_latex(index=False, escape=True,
                caption="Prescriptive Safety Audit metrics by model and cohort. PSS higher is safer; "
                        "DHI is mean positive net excess deaths per 1000 across the operating range.",
                label="tab:prescriptive"))
    logger.info("Table 3 written.")
else:
    table3 = pd.DataFrame(); logger.warning("prescriptive scores missing.")
table3


00:55:07 | INFO    | Table 3 written.


,model,cohort,PSS,DHI,DHI_frac,CAR_mean,concordance_r_b,PSS 95% CI,DHI 95% CI
0,logreg,MIMIC-hold,0.813,-10.382,0.171,0.837,0.336,,
1,random_forest,MIMIC-hold,0.832,-10.947,0.171,0.840,0.331,,
2,hist_gb,MIMIC-hold,0.821,-10.262,0.146,0.839,0.315,,
3,lightgbm,MIMIC-hold,0.821,-10.160,0.146,0.840,0.314,"[0.813, 0.829]","[-11.076, -9.195]"
4,xgboost,MIMIC-hold,0.822,-10.107,0.146,0.840,0.315,,
5,logreg,eICU-ext,0.856,-12.378,0.049,0.835,0.406,,
6,random_forest,eICU-ext,0.831,-11.494,0.073,0.805,0.404,,
7,hist_gb,eICU-ext,0.860,-12.373,0.024,0.833,0.379,,
8,lightgbm,eICU-ext,0.856,-12.163,0.049,0.829,0.380,"[0.853, 0.859]","[-12.594, -11.705]"
9,xgboost,eICU-ext,0.862,-12.437,0.024,0.836,0.376,,


In [14]:
# =============================================================================
# TABLE 4. External validation and transportability (internal vs external)
# =============================================================================
rows = []
if T["model_performance"] is not None:
    mp = T["model_performance"]
    for metric in ["AUROC", "Brier", "ECE"]:
        vi = mp[(mp.model == PRIMARY) & (mp.cohort == "MIMIC-hold")][metric]
        ve = mp[(mp.model == PRIMARY) & (mp.cohort == "eICU-ext")][metric]
        if len(vi) and len(ve):
            rows.append({"quantity": metric, "internal (MIMIC)": round(float(vi.iloc[0]), 3),
                         "external (eICU)": round(float(ve.iloc[0]), 3),
                         "change": round(float(ve.iloc[0]) - float(vi.iloc[0]), 3)})
for lab, si, se in [("PSS", sc_hold["PSS"], sc_ext["PSS"]),
                    ("DHI", sc_hold["DHI"], sc_ext["DHI"]),
                    ("CAR", sc_hold["CAR_mean"], sc_ext["CAR_mean"])]:
    rows.append({"quantity": lab, "internal (MIMIC)": round(si, 3),
                 "external (eICU)": round(se, 3), "change": round(se - si, 3)})
table4 = pd.DataFrame(rows)
table4.to_csv(CFG.TAB_DIR / "table4_external_validation.csv", index=False)
with open(CFG.TAB_DIR / "table4_external_validation.tex", "w") as f:
    f.write(table4.to_latex(index=False, escape=True,
            caption="Transportability of prediction and prescriptive-safety quantities from MIMIC-IV to eICU-CRD.",
            label="tab:external"))
logger.info("Table 4 written.")
table4


00:55:07 | INFO    | Table 4 written.


,quantity,internal (MIMIC),external (eICU),change
0,AUROC,0.909,0.822,-0.087
1,Brier,0.064,0.077,0.014
2,ECE,0.008,0.037,0.029
3,PSS,0.821,0.856,0.035
4,DHI,-10.161,-12.163,-2.002
5,CAR,0.840,0.829,-0.010


In [15]:
# =============================================================================
# P8: Transportability decomposition (covariate shift vs effect modification)
# =============================================================================
# Load design matrices saved by NB2
try:
    design_data = np.load(CFG.NB2_DIR / "cache" / "design_matrices.npz")
    X_dev = design_data["X_dev"]
    X_ext = design_data["X_ext"]
    y_dev = design_data["y_dev"]
    y_ext = design_data["y_ext"]
    a_dev = design_data["a_dev"]
    a_ext = design_data["a_ext"]
    logger.info("Loaded design matrices from NB2 cache.")
except Exception as e:
    logger.warning("Could not load design matrices: %s. Recomputing from analytic tables...")
    raise RuntimeError("Design matrices not available. Please run NB2 first and ensure cache is saved.") from e

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

def domain_shift(X_m, X_e, y_m, y_e, a_m, a_e, r_m, r_e, seed=CFG.RANDOM_SEED):
    """Decompose transportability into covariate shift vs effect modification."""
    # Train domain classifier
    X = np.vstack([X_m, X_e])
    s = np.r_[np.zeros(len(X_m)), np.ones(len(X_e))]
    clf = HistGradientBoostingClassifier(random_state=seed).fit(X, s)
    ps_site = np.clip(clf.predict_proba(X_e)[:,1], 1e-3, 1-1e-3)
    w = (1 - ps_site) / ps_site  # weight eICU toward MIMIC covariate distribution
    auc = roc_auc_score(s, clf.predict_proba(X)[:,1])
    
    # Unweighted external DHI
    sw_e = sweep(r_e, benefit_ext, a_e)
    dhi_ext = scalars(sw_e)["DHI_signed"]
    
    # Weighted external DHI (reweighted to MIMIC covariate distribution)
    # Note: sweep doesn't support weights natively; we approximate by reweighting EED
    eed_weighted = np.array([np.average(
        deployment_delta(r_e, benefit_ext, a_e, t)[0] / len(r_e) * 1000,
        weights=w
    ) for t in THR])
    dhi_ext_weighted = float(np.mean(eed_weighted))
    
    return {
        "site_auc": round(float(auc), 3),
        "dhi_ext_unweighted": round(dhi_ext, 3),
        "dhi_ext_weighted_to_mimic": round(dhi_ext_weighted, 3),
        "attributable_to_covariate_shift": round(dhi_ext - dhi_ext_weighted, 3),
        "n_shift": len(w)
    }

# Run decomposition
transport_results = domain_shift(X_dev, X_ext, y_dev, y_ext, a_dev, a_ext, r_hold, r_ext)
pd.DataFrame([transport_results]).to_csv(CFG.TAB_DIR / "transportability_decomposition.csv", index=False)
logger.info("Transportability decomposition: %s", transport_results)

00:55:08 | INFO    | Loaded design matrices from NB2 cache.


  File "C:\Users\<user>\anaconda3-v2\Lib\site-packages\joblib\externals\loky\backend\context.py", line 199, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\<user>\anaconda3-v2\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\<user>\anaconda3-v2\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\<user>\anaconda3-v2\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


00:55:23 | INFO    | Transportability decomposition: {'site_auc': 1.0, 'dhi_ext_unweighted': -12.163, 'dhi_ext_weighted_to_mimic': -0.0, 'attributable_to_covariate_shift': -12.163, 'n_shift': 86403}


In [16]:
# =============================================================================
# TABLE 5. Sensitivity and robustness analyses
# =============================================================================
parts = []
if T["robustness_battery"] is not None:
    rb = T["robustness_battery"].copy(); rb.insert(0, "analysis", "robustness"); parts.append(rb)
if T["ablation_mitigation"] is not None:
    ab = T["ablation_mitigation"].copy()
    ab = ab.rename(columns={"model": "scenario"}); ab.insert(0, "analysis", "ablation")
    parts.append(ab)
table5 = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
if evalue:
    table5 = pd.concat([table5, pd.DataFrame([{
        "analysis": "unmeasured confounding", "scenario": "E-value (ATE)",
        "PSS": np.nan, "DHI": np.nan, "CAR_mean": np.nan,
        "note": f"RR {evalue.get('risk_ratio', float('nan')):.2f}, E-value {evalue.get('E_value', float('nan')):.2f}"}])],
        ignore_index=True)
table5.to_csv(CFG.TAB_DIR / "table5_sensitivity.csv", index=False)
logger.info("Table 5 written (%d rows).", len(table5))
table5


00:55:23 | INFO    | Table 5 written (10 rows).


,analysis,scenario,PSS,DHI,DHI_frac,CAR_mean,TII,AUROC,n_surrogate_features,mechanism_present,note
0,robustness,reference (MIMIC hold),0.8212,-10.1605,0.1463,0.8395,NaN,NaN,NaN,NaN,NaN
1,robustness,external (eICU),0.8557,-12.1630,0.0488,0.8293,NaN,NaN,NaN,NaN,NaN
2,robustness,treatment prevalence 30%,0.7884,-6.7204,0.2195,0.8535,NaN,NaN,NaN,NaN,NaN
3,robustness,treatment prevalence 60%,0.8926,-21.3651,0.0488,0.7953,NaN,NaN,NaN,NaN,NaN
4,robustness,added 20% missingness,0.8444,-12.5109,0.1220,0.8623,NaN,NaN,NaN,NaN,NaN
5,robustness,eICU no recalibration,0.8566,-12.6728,0.0488,0.8274,NaN,NaN,NaN,NaN,NaN
6,robustness,eICU site recalibrated,0.8312,-12.3202,0.0976,0.8256,NaN,NaN,NaN,NaN,NaN
7,ablation,full covariates,0.8212,-10.1605,NaN,NaN,0.0508,0.9088,0.0,False,NaN
8,ablation,surrogates removed,0.8212,-10.1605,NaN,NaN,0.0508,0.9088,0.0,False,NaN
9,unmeasured confounding,E-value (ATE),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"RR 1.78, E-value 2.96"


In [17]:
# =============================================================================
# TABLE 6. Subgroup analyses + bootstrap CIs + BH multiplicity control
# =============================================================================
def boot_subgroup(r, benefit, a, B=CFG.N_BOOTSTRAP, seed=CFG.RANDOM_SEED):
    """One resampling pass; returns DHI, PSS, and signed net-EED draws."""
    rng = np.random.default_rng(seed); n = len(r)
    dhi, pss, net = [], [], []
    for _ in range(B):
        idx = rng.integers(0, n, n)
        sw = sweep(r[idx], benefit[idx], a[idx]); sc = scalars(sw)
        dhi.append(sc["DHI"]); pss.append(sc["PSS"])
        net.append(float(np.mean(sw["eed_net"])))   # signed net excess deaths/1000, can be < 0
    return np.asarray(dhi), np.asarray(pss), np.asarray(net)

net_draws = {}          # keyed by "variable=level" so BH spans every tested subgroup
if HAVE_HOLD_META:
    with timed("subgroup bootstrap CIs"):
        rows = []
        for col in ["sex", "race_group", "age_band"]:
            for level, grp in hold_meta.groupby(col, observed=True):
                mask = (hold_meta[col] == level).values
                if mask.sum() < 40:
                    continue
                d, p, ne = boot_subgroup(r_hold[mask], benefit_hold[mask], a_hold[mask])
                key = f"{col}={level}"
                net_draws[key] = ne
                rows.append({"variable": col, "level": str(level), "n": int(mask.sum()),
                             "PSS": round(float(p.mean()), 3),
                             "PSS_CI": f"[{np.percentile(p,2.5):.3f}, {np.percentile(p,97.5):.3f}]",
                             "DHI": round(float(d.mean()), 2),
                             "DHI_CI": f"[{np.percentile(d,2.5):.2f}, {np.percentile(d,97.5):.2f}]",
                             "_key": key})
        table6 = pd.DataFrame(rows)
elif T["subgroup_prescriptive_safety"] is not None:
    table6 = T["subgroup_prescriptive_safety"].copy()
else:
    table6 = pd.DataFrame()

# --- BH-adjusted one-sided test that net excess deaths > 0 (only if we bootstrapped) ---
if net_draws:
    keys = list(net_draws.keys())
    pvals = [min(max(float(np.mean(net_draws[k] <= 0)), 1e-4), 1.0) for k in keys]
    try:
        from statsmodels.stats.multitest import multipletests
        rej, p_adj, _, _ = multipletests(pvals, method="fdr_bh")
    except Exception:                       # manual BH fallback
        order = np.argsort(pvals); m = len(pvals); p_adj = np.empty(m)
        for rank, idx in enumerate(order, 1): p_adj[idx] = min(1.0, pvals[idx]*m/rank)
        rej = p_adj < 0.05
    sig = pd.DataFrame({"_key": keys, "p_raw": np.round(pvals, 4),
                        "p_BH": np.round(p_adj, 4), "survives_BH": rej})
    sig.to_csv(CFG.TAB_DIR / "subgroup_multiplicity.csv", index=False)
    table6 = table6.merge(sig, on="_key", how="left")

if "_key" in table6.columns:
    table6 = table6.drop(columns="_key")

table6.to_csv(CFG.TAB_DIR / "table6_subgroups.csv", index=False)
with open(CFG.TAB_DIR / "table6_subgroups.tex", "w") as f:
    f.write(table6.to_latex(index=False, escape=True,
            caption="Prescriptive harm across demographic subgroups with bootstrap 95% intervals and BH-adjusted significance.",
            label="tab:subgroups"))
logger.info("Table 6 written (%d rows).", len(table6))
table6

00:55:23 | INFO    | START  subgroup bootstrap CIs
00:56:20 | INFO    | DONE   subgroup bootstrap CIs  (57.2s)
00:56:20 | INFO    | Table 6 written (11 rows).


,variable,level,n,PSS,PSS_CI,DHI,DHI_CI,p_raw,p_BH,survives_BH
0,sex,Female,4686,0.804,"[0.791, 0.817]",-9.55,"[-10.88, -8.22]",1.000,1.0,False
1,sex,Male,6135,0.834,"[0.823, 0.844]",-10.67,"[-11.91, -9.34]",1.000,1.0,False
2,race_group,Asian,308,0.731,"[0.655, 0.795]",-6.78,"[-12.88, -0.11]",0.976,1.0,False
3,race_group,Black,967,0.836,"[0.803, 0.865]",-10.30,"[-13.44, -6.79]",1.000,1.0,False
4,race_group,Hispanic,343,0.860,"[0.805, 0.901]",-12.15,"[-17.01, -8.40]",1.000,1.0,False
5,race_group,Other/Unknown,2008,0.836,"[0.817, 0.856]",-14.04,"[-16.36, -11.67]",1.000,1.0,False
6,race_group,White,7195,0.817,"[0.807, 0.829]",-9.14,"[-10.46, -7.86]",1.000,1.0,False
7,age_band,<50,1964,0.919,"[0.902, 0.936]",-12.80,"[-14.99, -10.81]",1.000,1.0,False
8,age_band,50-64,3014,0.850,"[0.833, 0.866]",-10.90,"[-12.67, -8.83]",1.000,1.0,False
9,age_band,65-79,3704,0.832,"[0.818, 0.846]",-11.69,"[-13.24, -10.16]",1.000,1.0,False


In [18]:
# =============================================================================
# Assemble all main tables into a single Excel workbook
# =============================================================================
main_tables = {"T1_baseline": table1, "T2_performance": table2, "T3_prescriptive": table3,
               "T4_external": table4, "T5_sensitivity": table5, "T6_subgroups": table6}
xlsx_path = CFG.TAB_DIR / "all_main_tables.xlsx"
try:
    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as xw:
        for sheet, df in main_tables.items():
            (df if df is not None and not df.empty else pd.DataFrame({"note": ["not available"]})
             ).to_excel(xw, sheet_name=sheet[:31], index=False)
    logger.info("Excel workbook written -> %s", xlsx_path.name)
except Exception as ex:
    logger.warning("Excel export skipped (%s); CSVs already written.", ex)
print("Main tables assembled:", ", ".join(main_tables))


00:56:21 | INFO    | Excel workbook written -> all_main_tables.xlsx
Main tables assembled: T1_baseline, T2_performance, T3_prescriptive, T4_external, T5_sensitivity, T6_subgroups


---
## C. Inferential hardening

Everything to this point reports point estimates and bootstrap intervals. A top-tier reviewer will ask three further questions, and this section answers each with a dedicated analysis.

1. *Is the measured prescriptive harm distinguishable from chance?* We answer with randomisation inference: permutation nulls that break the link between predicted risk and estimated benefit and between the score and the patients, giving an exact p-value for the deployment hazard.
2. *Does the causal pipeline return a null when it should, and is the treatment-effect model itself any good?* We answer with a negative-control outcome that treatment cannot cause, and with the c-for-benefit and benefit-decile calibration for the treatment-effect model that the whole audit rests on.
3. *How fragile is the conclusion?* We propagate treatment-effect estimation error into the hazard metrics, and we compute a tipping point: how large a systematic bias in the estimated benefit would have to be to overturn the safety verdict.

These use the cached vectors and a small, dependency-light recomputation on the analytic tables. No production models are retrained.

In [19]:
# =============================================================================
# Compact, dependency-light modelling utilities for the NB3-internal analyses
# (histogram gradient boosting only; used for negative controls and CATE bootstrap)
# =============================================================================
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

ID_COLS = ["db", "patient_id", "hadm_id"]
OUTCOME_COLS = ["in_hospital_mortality", "icu_mortality", "mortality_28d", "hosp_los_days"]
TREAT_COLS = ["vaso_post", "vaso_day1", "inotrope_post", "vent_post", "vent_day1",
              "invasive_vent_post", "rrt_post", "rrt_day1", "n_pressor_types_post",
              "any_organ_support_post"]
CAT_COLS = [c for c in ["sex", "race_group", "admission_type", "insurance"] if c in mimic.columns]
_excl = set(ID_COLS + OUTCOME_COLS + TREAT_COLS + ["los_h", "sofa_coverage"])
NUM_COLS = [c for c in mimic.columns if c not in _excl and c not in CAT_COLS
            and pd.api.types.is_numeric_dtype(mimic[c])]
FEATURES = NUM_COLS + CAT_COLS

def make_pre():
    num = Pipeline([("impute", SimpleImputer(strategy="median", add_indicator=True))])
    cat = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                    ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
    return ColumnTransformer([("num", num, NUM_COLS), ("cat", cat, CAT_COLS)],
                             remainder="drop", verbose_feature_names_out=False)

# reproduce dev/hold consistently with the verified reconstruction
mimic_dev, mimic_hold2 = train_test_split(mimic, test_size=0.30, random_state=CFG.RANDOM_SEED,
                                          stratify=mimic[CFG.OUTCOME])
mimic_dev = mimic_dev.reset_index(drop=True); mimic_hold2 = mimic_hold2.reset_index(drop=True)
logger.info("NB3 modelling utilities ready (%d features).", len(FEATURES))


00:56:21 | INFO    | NB3 modelling utilities ready (177 features).


In [20]:
# =============================================================================
# C1. Randomisation (permutation) inference for the deployment hazard
# =============================================================================
def dhi_from(r, benefit, a):
    return scalars(sweep(r, benefit, a))["DHI"]

obs_dhi = sc_hold["DHI"]
rng = np.random.default_rng(CFG.RANDOM_SEED)
P = CFG.N_PERMUTE if not CFG.FAST else min(CFG.N_PERMUTE, 800)

with timed("permutation inference"):
    null_benefit = np.empty(P); null_risk = np.empty(P)
    for k in range(P):
        null_benefit[k] = dhi_from(r_hold, rng.permutation(benefit_hold), a_hold)
        null_risk[k]    = dhi_from(rng.permutation(r_hold), benefit_hold, a_hold)
p_benefit = (1 + np.sum(null_benefit >= obs_dhi)) / (P + 1)
p_risk    = (1 + np.sum(null_risk >= obs_dhi)) / (P + 1)
perm = {"observed_DHI": obs_dhi, "p_benefit_permutation": float(p_benefit),
        "p_risk_permutation": float(p_risk),
        "null_benefit_mean": float(null_benefit.mean()), "null_risk_mean": float(null_risk.mean())}
json.dump(perm, open(CFG.TAB_DIR / "permutation_inference.json", "w"), indent=2)
logger.info("Permutation p-values: benefit-null %.4f | risk-null %.4f", p_benefit, p_risk)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
for ax, null, lab, pv in [(axes[0], null_benefit, "benefit permuted", p_benefit),
                          (axes[1], null_risk, "risk permuted", p_risk)]:
    ax.hist(null, bins=40, color=PALETTE["neutral"], alpha=0.8)
    ax.axvline(obs_dhi, color=PALETTE["harm"], lw=2, label=f"observed DHI = {obs_dhi:.2f}")
    ax.set_title(f"Null: {lab}\npermutation p = {pv:.4f}")
    ax.set_xlabel("DHI under null"); ax.set_ylabel("frequency"); ax.legend(fontsize=8)
savefig(fig, "fig_permutation_inference")
perm


00:56:21 | INFO    | START  permutation inference
00:56:59 | INFO    | DONE   permutation inference  (37.8s)
00:56:59 | INFO    | Permutation p-values: benefit-null 0.0012 | risk-null 0.0012
00:57:00 | INFO    | figure -> fig_permutation_inference


{'observed_DHI': -10.160543169288736,
 'p_benefit_permutation': 0.0012484394506866417,
 'p_risk_permutation': 0.0012484394506866417,
 'null_benefit_mean': -13.41558549028432,
 'null_risk_mean': -11.897285423028011}

In [21]:
# =============================================================================
# C2. Negative-control outcome: treatment cannot cause a fixed baseline trait
# A valid pipeline should return an adjusted effect near zero even though the
# naive (unadjusted) association is biased by confounding by indication.
# =============================================================================
def g_formula_ate(df, outcome, seed=CFG.RANDOM_SEED):
    pre = make_pre(); X = pre.fit_transform(df[FEATURES])
    a = df[CFG.TREATMENT].astype(int).values; yv = df[outcome].astype(int).values
    m1 = HistGradientBoostingClassifier(max_iter=150, random_state=seed).fit(X[a == 1], yv[a == 1])
    m0 = HistGradientBoostingClassifier(max_iter=150, random_state=seed).fit(X[a == 0], yv[a == 0])
    return float(np.mean(m1.predict_proba(X)[:, 1] - m0.predict_proba(X)[:, 1]))

def g_formula_ate_exposure(df, exposure_col, outcome_col, seed=CFG.RANDOM_SEED):
    pre = make_pre(); X = pre.fit_transform(df[FEATURES])
    a = df[exposure_col].astype(int).values; yv = df[outcome_col].astype(int).values
    if a.sum() < 20 or (1 - a).sum() < 20:
        return float("nan")
    m1 = HistGradientBoostingClassifier(max_iter=150, random_state=seed).fit(X[a == 1], yv[a == 1])
    m0 = HistGradientBoostingClassifier(max_iter=150, random_state=seed).fit(X[a == 0], yv[a == 0])
    return float(np.mean(m1.predict_proba(X)[:, 1] - m0.predict_proba(X)[:, 1]))

# construct a negative-control outcome that is immutable to post-anchor treatment
nco_defs = {}
if "admission_age" in mimic.columns:
    mimic = mimic.copy(); mimic["nco_age_over_median"] = (mimic["admission_age"] > mimic["admission_age"].median()).astype(int)
    nco_defs["age_over_median (immutable)"] = "nco_age_over_median"
if "sex" in mimic.columns:
    mimic["nco_sex_female"] = (mimic["sex"].astype(str).str.lower().str.startswith("f")).astype(int)
    nco_defs["sex_female (immutable)"] = "nco_sex_female"

with timed("negative-control outcomes"):
    nc_rows = []
    for lab, col in nco_defs.items():
        naive = float(mimic.loc[mimic[CFG.TREATMENT] == 1, col].mean()
                      - mimic.loc[mimic[CFG.TREATMENT] == 0, col].mean())
        adj = g_formula_ate(mimic, col)
        nc_rows.append({"negative_control_outcome": lab, "naive_association": round(naive, 3),
                        "adjusted_effect": round(adj, 3)})
    # positive reference: the real outcome should show a non-null adjusted effect
    real_adj = g_formula_ate(mimic, CFG.OUTCOME)
    nc_rows.append({"negative_control_outcome": f"{CFG.OUTCOME} (real, reference)",
                    "naive_association": round(float(mimic.loc[mimic[CFG.TREATMENT]==1, CFG.OUTCOME].mean()
                                        - mimic.loc[mimic[CFG.TREATMENT]==0, CFG.OUTCOME].mean()), 3),
                    "adjusted_effect": round(real_adj, 3)})

    # negative-control EXPOSURE: a random sham treatment with no causal path to the outcome
    rng = np.random.default_rng(CFG.RANDOM_SEED)
    mimic["sham_exposure"] = rng.binomial(1, float(mimic[CFG.TREATMENT].mean()), len(mimic))
    sham_adj = g_formula_ate_exposure(mimic, "sham_exposure", CFG.OUTCOME)
    nc_rows.append({"negative_control_outcome": "sham exposure (random, no causal path)",
                    "naive_association": round(float(mimic.loc[mimic.sham_exposure==1, CFG.OUTCOME].mean()
                                        - mimic.loc[mimic.sham_exposure==0, CFG.OUTCOME].mean()), 3),
                    "adjusted_effect": round(sham_adj, 3)})
neg_controls = pd.DataFrame(nc_rows)
neg_controls.to_csv(CFG.TAB_DIR / "negative_controls.csv", index=False)
logger.info("Negative controls computed (adjusted effect on immutable traits should be near 0).")

fig, ax = plt.subplots(figsize=(8, 4.2))
yy = np.arange(len(neg_controls))
ax.scatter(neg_controls["naive_association"], yy, color=PALETTE["neutral"], label="naive association", s=60)
ax.scatter(neg_controls["adjusted_effect"], yy, color=PALETTE["harm"], label="adjusted effect", s=60)
ax.axvline(0, color="k", lw=0.8)
ax.set_yticks(yy); ax.set_yticklabels(neg_controls["negative_control_outcome"], fontsize=8)
ax.set_xlabel("effect on outcome (risk difference)"); ax.legend(fontsize=8)
ax.set_title("Negative-control outcomes: adjustment removes spurious association"); ax.invert_yaxis()
savefig(fig, "fig_negative_controls")
neg_controls


00:57:00 | INFO    | START  negative-control outcomes
00:57:44 | INFO    | DONE   negative-control outcomes  (43.8s)
00:57:44 | INFO    | Negative controls computed (adjusted effect on immutable traits should be near 0).
00:57:44 | INFO    | figure -> fig_negative_controls


,negative_control_outcome,naive_association,adjusted_effect
0,age_over_median (immutable),-0.004,-0.000
1,sex_female (immutable),-0.020,-0.000
2,"in_hospital_mortality (real, reference)",0.170,0.065
3,"sham exposure (random, no causal path)",0.003,0.001


In [22]:
# =============================================================================
# C3. Treatment-effect model validation: c-for-benefit and benefit calibration
# The audit rests on the estimated benefit, so we test that it discriminates
# (c-for-benefit > 0.5) and calibrates (predicted vs observed benefit).
#
# Pairing: 1:1 nearest-neighbour matching of treated to control on the predicted
# RISK score (a prognostic/balancing score summarising L), rather than pairing
# independently rank-sorted arms. This follows van Klaveren (2018) / Hoogland
# (2024). Matching on the risk score (not the benefit being validated) avoids
# circularity. Under confounding these observational estimates are biased in a
# non-trivial way; see Xia, Sadatsafavi & Gustafson (AJE 2026).
# =============================================================================
from sklearn.neighbors import NearestNeighbors

def matched_c_for_benefit(pred_benefit, match_key, a, y, seed=CFG.RANDOM_SEED, bootstrap=True, B=300):
    """c-for-benefit with bootstrap CI and interpretability gate."""
    ti = np.where(a == 1)[0]; ci = np.where(a == 0)[0]
    if len(ti) < 10 or len(ci) < 10:
        return np.nan, None, None, {"c_for_benefit": np.nan, "ci_lo": np.nan, "ci_hi": np.nan,
                                    "interpretable_individual_level": False}

    key = np.asarray(match_key, dtype=float).reshape(-1, 1)
    nn = NearestNeighbors(n_neighbors=1).fit(key[ci])
    _, j = nn.kneighbors(key[ti]); cpart = ci[j.ravel()]
    pair_pred = 0.5 * (pred_benefit[ti] + pred_benefit[cpart])
    pair_obs = (y[cpart] - y[ti]).astype(float)
    m = len(pair_pred)
    rng = np.random.default_rng(seed)

    def _compute_conc():
        K = min(60000, m * m)
        i, jj = rng.integers(0, m, K), rng.integers(0, m, K)
        ok = (pair_pred[i] != pair_pred[jj]) & (pair_obs[i] != pair_obs[jj])
        if ok.sum() == 0:
            return 0.5
        return float(((pair_pred[i] > pair_pred[jj]) == (pair_obs[i] > pair_obs[jj]))[ok].mean())

    base = _compute_conc()

    if bootstrap:
        boot_vals = []
        n = len(pair_pred)
        for b in range(B):
            idx = rng.integers(0, n, n)
            p_pred_b = pair_pred[idx]; p_obs_b = pair_obs[idx]
            K = min(60000, n * n)
            i, jj = rng.integers(0, n, K), rng.integers(0, n, K)
            ok = (p_pred_b[i] != p_pred_b[jj]) & (p_obs_b[i] != p_obs_b[jj])
            if ok.sum() > 0:
                boot_vals.append(float(((p_pred_b[i] > p_pred_b[jj]) == (p_obs_b[i] > p_obs_b[jj]))[ok].mean()))
            else:
                boot_vals.append(0.5)
        lo, hi = np.percentile(boot_vals, [2.5, 97.5])
        interpretable = bool(lo > 0.5)
    else:
        lo, hi = base, base
        interpretable = bool(base > 0.5)

    gate = {"c_for_benefit": round(base, 3), "ci_lo": round(float(lo), 3),
            "ci_hi": round(float(hi), 3), "interpretable_individual_level": interpretable}

    return base, pair_pred, pair_obs, gate

# match on the risk score r_* (prognostic/balancing score)
cfb_hold, pp, po, gate_hold = matched_c_for_benefit(benefit_hold, r_hold, a_hold, y_hold)
cfb_ext,  _,  _,  gate_ext  = matched_c_for_benefit(benefit_ext,  r_ext,  a_ext,  y_ext)
logger.info("Matched c-for-benefit: hold %.3f | ext %.3f (0.5 = no discrimination of benefit)",
            cfb_hold, cfb_ext)

# ---- benefit-validation gate: propagate an interpretability flag downstream ----
gate_hold["cohort"] = "MIMIC-hold"; gate_ext["cohort"] = "eICU-ext"
pd.DataFrame([gate_hold, gate_ext]).to_csv(CFG.TAB_DIR / "benefit_gate.csv", index=False)
GATE_PASSED = bool(gate_hold["interpretable_individual_level"])
PRESCRIPTIVE_STATUS = ("individual-level finding" if GATE_PASSED else
                       "illustrative only: benefit does not discriminate "
                       f"(c-for-benefit {cfb_hold:.2f}, 95% CI "
                       f"[{gate_hold['ci_lo']}, {gate_hold['ci_hi']}] includes 0.5)")
logger.info("Benefit gate: %s", PRESCRIPTIVE_STATUS)

# ---- calibration-for-benefit: Eavg / E50 / E90 and ICI on matched pairs ----
benefit_cal_metrics = {"Eavg_for_benefit": float("nan"), "E50_for_benefit": float("nan"),
                       "E90_for_benefit": float("nan"), "ICI_for_benefit": float("nan")}
smooth = None
if pp is not None and len(pp) >= 20:
    order = np.argsort(pp)
    pp_s, po_s = pp[order], po[order]
    try:
        import statsmodels.api as sm
        smooth = sm.nonparametric.lowess(po_s, pp_s, frac=0.5, return_sorted=False)
    except Exception:
        k = max(10, len(pp_s) // 20)
        smooth = np.convolve(po_s, np.ones(k) / k, mode="same")   # fallback moving average
    abs_err = np.abs(pp_s - smooth)
    benefit_cal_metrics = {"Eavg_for_benefit": float(np.mean(abs_err)),   # = ICI-for-benefit
                           "E50_for_benefit":  float(np.median(abs_err)),
                           "E90_for_benefit":  float(np.percentile(abs_err, 90)),
                           "ICI_for_benefit":  float(np.mean(abs_err))}
json.dump(benefit_cal_metrics, open(CFG.TAB_DIR / "benefit_calibration_metrics.json", "w"), indent=2)
logger.info("Benefit calibration Eavg %.3f | E90 %.3f | c-for-benefit(matched) %.3f",
            benefit_cal_metrics["Eavg_for_benefit"], benefit_cal_metrics["E90_for_benefit"], cfb_hold)

# ---- decile calibration table on the hold-out (kept for the figure) ----
dec = pd.qcut(benefit_hold, 10, labels=False, duplicates="drop")
cal_rows = []
for d in np.unique(dec):
    mmask = dec == d
    if (a_hold[mmask] == 1).sum() >= 5 and (a_hold[mmask] == 0).sum() >= 5:
        obs = y_hold[mmask & (a_hold == 0)].mean() - y_hold[mmask & (a_hold == 1)].mean()
        cal_rows.append({"decile": int(d), "predicted_benefit": float(benefit_hold[mmask].mean()),
                         "observed_benefit": float(obs), "n": int(mmask.sum())})
benefit_cal = pd.DataFrame(cal_rows)
benefit_cal.to_csv(CFG.TAB_DIR / "benefit_calibration.csv", index=False)
json.dump({"c_for_benefit_hold": float(cfb_hold), "c_for_benefit_ext": float(cfb_ext),
           "matching": "1:1 nearest-neighbour on predicted risk score (prognostic score)"},
          open(CFG.TAB_DIR / "c_for_benefit.json", "w"), indent=2)

# ---- figure: benefit distribution + matched calibration curve (with decile overlay) ----
fig, axes = plt.subplots(1, 2, figsize=(12.5, 5))
axes[0].hist(benefit_hold, bins=40, color=PALETTE["accent"], alpha=0.85)
axes[0].axvline(DELTA, color=PALETTE["harm"], ls="--", lw=1, label=f"benefit margin delta={DELTA}")
axes[0].set_title(f"Estimated benefit distribution\nmatched c-for-benefit = {cfb_hold:.3f}")
axes[0].set_xlabel("estimated benefit b = -CATE"); axes[0].set_ylabel("patients"); axes[0].legend(fontsize=8)

if smooth is not None:
    axes[1].plot(pp_s, smooth, "-", color=PALETTE["MIMIC"], lw=1.8,
                 label=f"matched (loess); Eavg={benefit_cal_metrics['Eavg_for_benefit']:.3f}")
if not benefit_cal.empty:
    axes[1].plot(benefit_cal["predicted_benefit"], benefit_cal["observed_benefit"], "o",
                 color=PALETTE["neutral"], ms=5, alpha=0.8, label="decile means")
    lo = float(min(benefit_cal[["predicted_benefit", "observed_benefit"]].min().min(),
                   pp.min() if pp is not None else 0)) - 0.02
    hi = float(max(benefit_cal[["predicted_benefit", "observed_benefit"]].max().max(),
                   pp.max() if pp is not None else 0)) + 0.02
    axes[1].plot([lo, hi], [lo, hi], "k--", lw=0.8, label="ideal")
axes[1].set_title("Benefit calibration (matched pairs)")
axes[1].set_xlabel("predicted benefit"); axes[1].set_ylabel("observed benefit (control - treated mortality)")
axes[1].legend(fontsize=8)
savefig(fig, "fig_benefit_validation")
benefit_cal

00:57:46 | INFO    | Matched c-for-benefit: hold 0.565 | ext 0.524 (0.5 = no discrimination of benefit)
00:57:46 | INFO    | Benefit gate: individual-level finding
00:57:48 | INFO    | Benefit calibration Eavg 0.084 | E90 0.158 | c-for-benefit(matched) 0.565
00:57:50 | INFO    | figure -> fig_benefit_validation


,decile,predicted_benefit,observed_benefit,n
0,0,-0.230747,-0.250777,1083
1,1,-0.089378,-0.276571,1084
2,2,-0.069921,-0.246566,1080
3,3,-0.058999,-0.169129,1083
4,4,-0.052265,-0.105805,1106
5,5,-0.045263,-0.080144,1191
6,6,-0.041337,-0.077674,1035
7,7,-0.041300,-0.029607,2138
8,8,0.052590,-0.197458,1021


In [23]:
def g_formula_ate_exposure(df, exposure_col, outcome_col, seed=CFG.RANDOM_SEED):
    pre = make_pre(); X = pre.fit_transform(df[FEATURES])
    a = df[exposure_col].astype(int).values; yv = df[outcome_col].astype(int).values
    if a.sum() < 20 or (1 - a).sum() < 20:
        return float("nan")
    m1 = HistGradientBoostingClassifier(max_iter=150, random_state=seed).fit(X[a == 1], yv[a == 1])
    m0 = HistGradientBoostingClassifier(max_iter=150, random_state=seed).fit(X[a == 0], yv[a == 0])
    return float(np.mean(m1.predict_proba(X)[:, 1] - m0.predict_proba(X)[:, 1]))

In [24]:
# =============================================================================
# C4. Uncertainty propagation: treatment-effect-model error into the hazard
# Bootstrap the T-learner (light HGB) on the development split, re-estimate the
# benefit on the hold-out, and recompute the DHI, so intervals carry causal-model
# uncertainty rather than patient sampling alone.
# =============================================================================
def _dr_predict(Xtr, ytr, atr, Xeval, seed=0, folds=2, mit=60, cap=12000):
    from sklearn.model_selection import StratifiedKFold
    from sklearn.ensemble import HistGradientBoostingClassifier as _HGBc
    from sklearn.ensemble import HistGradientBoostingRegressor as _HGBr
    if len(ytr) > cap:                                   # subsample per draw for tractability
        s = np.random.default_rng(seed).choice(len(ytr), cap, replace=False)
        Xtr, ytr, atr = Xtr[s], ytr[s], atr[s]
    kf = StratifiedKFold(folds, shuffle=True, random_state=seed)
    mu1 = np.zeros(len(ytr)); mu0 = np.zeros(len(ytr)); e = np.zeros(len(ytr))
    for tr, te in kf.split(Xtr, ytr):
        e[te] = np.clip(_HGBc(max_iter=mit, random_state=seed)
                        .fit(Xtr[tr], atr[tr]).predict_proba(Xtr[te])[:, 1], 1e-3, 1-1e-3)
        t1 = tr[atr[tr] == 1]; t0 = tr[atr[tr] == 0]
        if len(t1) > 10 and len(t0) > 10:
            mu1[te] = _HGBc(max_iter=mit, random_state=seed).fit(Xtr[t1], ytr[t1]).predict_proba(Xtr[te])[:, 1]
            mu0[te] = _HGBc(max_iter=mit, random_state=seed).fit(Xtr[t0], ytr[t0]).predict_proba(Xtr[te])[:, 1]
        else:
            mu1[te] = ytr.mean(); mu0[te] = ytr.mean()
    pseudo = (mu1 - mu0) + atr*(ytr - mu1)/e - (1 - atr)*(ytr - mu0)/(1 - e)
    return _HGBr(max_iter=150, random_state=seed).fit(Xtr, pseudo).predict(Xeval)

cache_np = CFG.OUT / "cate_bootstrap.npz"
if False:  # force recompute of the DR uncertainty band (bypass stale cache)
    dhi_boot = np.load(cache_np)["dhi_boot"]
    logger.info("loaded CATE bootstrap from cache")
else:
    import gc
    dhi_boot = []
    with timed("treatment-effect-model bootstrap"):
        try:
            pre = make_pre(); pre.fit(mimic_dev[FEATURES])
            Xdev = pre.transform(mimic_dev[FEATURES]).astype(np.float32)
            Xhold = pre.transform(mimic_hold2[FEATURES]).astype(np.float32)
            adev = mimic_dev[CFG.TREATMENT].astype(int).values
            ydev = mimic_dev[CFG.OUTCOME].astype(int).values
            B = min(CFG.CATE_BOOT, 60)          # 60 draws is enough for a 95% band; keeps runtime ~7 min
            rng = np.random.default_rng(CFG.RANDOM_SEED)
            import time as _time; _t0 = _time.time()
            for _bi in range(B):
                try:
                    if _bi % 10 == 0:
                        print(f"  bootstrap draw {_bi}/{B} | {(_time.time()-_t0)/60:.1f} min", flush=True)
                    idx = rng.integers(0, len(Xdev), len(Xdev))
                    Xi, ai, yi = Xdev[idx], adev[idx], ydev[idx]
                    if ai.sum() < 20 or (1 - ai).sum() < 20:
                        continue
                    b_boot = -_dr_predict(Xi, yi, ai, Xhold, seed=0)
                    dhi_boot.append(scalars(sweep(r_hold, b_boot, a_hold))["DHI"])
                    del b_boot; gc.collect()
                except Exception as ex:
                    logger.warning("bootstrap draw skipped (%s)", ex)
        except Exception as ex:
            logger.warning("CATE bootstrap unavailable (%s); intervals fall back to patient sampling", ex)
        dhi_boot = np.array(dhi_boot, dtype=float)
        if dhi_boot.size:
            np.savez(cache_np, dhi_boot=dhi_boot)

# patient-level bootstrap for HRR and EED at the reference threshold
rng = np.random.default_rng(CFG.RANDOM_SEED); n = len(r_hold); hrr_b, eed_b = [], []
for _ in range(CFG.N_BOOTSTRAP):
    idx = rng.integers(0, n, n)
    hrr, hfr, en, eh, car = metrics_at(r_hold[idx], benefit_hold[idx], a_hold[idx], tau_ref)
    hrr_b.append(hrr); eed_b.append(en)
unc = {"DHI_causal_model": [float(np.mean(dhi_boot)), float(np.percentile(dhi_boot, 2.5)),
                            float(np.percentile(dhi_boot, 97.5))] if len(dhi_boot) else [np.nan]*3,
       "HRR_at_tau_ref": [float(np.mean(hrr_b)), float(np.percentile(hrr_b, 2.5)), float(np.percentile(hrr_b, 97.5))],
       "EEDnet_at_tau_ref": [float(np.mean(eed_b)), float(np.percentile(eed_b, 2.5)), float(np.percentile(eed_b, 97.5))]}
# confounding-inclusive band: shrink benefit toward null over a Gamma range and report DHI envelope
gammas = np.linspace(0.0, 1.0, 11)          # 1.0 = full nullification of the (possibly confounded) benefit
dhi_gamma = [scalars(sweep(r_hold, benefit_hold * (1 - g), a_hold))["DHI"] for g in gammas]
unc["DHI_confounding_band"] = [float(min(dhi_gamma)), float(max(dhi_gamma))]
json.dump(unc, open(CFG.TAB_DIR / "uncertainty_propagation.json", "w"), indent=2)
logger.info("DHI confounding-inclusive band: [%.2f, %.2f]", *unc["DHI_confounding_band"])

json.dump(unc, open(CFG.TAB_DIR / "uncertainty_propagation.json", "w"), indent=2)
logger.info("DHI incl. causal-model error: %.2f [%.2f, %.2f]", *unc["DHI_causal_model"])

fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
for ax, data, lab, obs in [(axes[0], dhi_boot, "DHI (causal-model bootstrap)", obs_dhi),
                           (axes[1], np.array(hrr_b)*100, "HRR % at tau_ref", np.mean(hrr_b)*100),
                           (axes[2], eed_b, "net EED /1000 at tau_ref", np.mean(eed_b))]:
    if len(data):
        ax.hist(data, bins=30, color=PALETTE["MIMIC"], alpha=0.8)
        ax.axvline(obs, color=PALETTE["harm"], lw=2)
    ax.set_title(lab); ax.set_ylabel("frequency")
savefig(fig, "fig_uncertainty")
unc


00:57:50 | INFO    | START  treatment-effect-model bootstrap
  bootstrap draw 0/40 | 0.0 min
  bootstrap draw 10/40 | 6.4 min
  bootstrap draw 20/40 | 12.6 min
  bootstrap draw 30/40 | 18.9 min
01:23:12 | INFO    | DONE   treatment-effect-model bootstrap  (1522.3s)
01:23:12 | INFO    | DHI confounding-inclusive band: [-10.16, 0.00]
01:23:12 | INFO    | DHI incl. causal-model error: -13.48 [-18.17, -6.20]
01:23:14 | INFO    | figure -> fig_uncertainty


{'DHI_causal_model': [-13.477013124701267,
  -18.173624205103078,
  -6.203861344723167],
 'HRR_at_tau_ref': [0.09210183901672672,
  0.08648923389705203,
  0.09782136586267443],
 'EEDnet_at_tau_ref': [-0.23863259536309198,
  -1.4257300738220904,
  1.0495944312735728],
 'DHI_confounding_band': [-10.160543169288736, 0.0]}

In [25]:
# =============================================================================
# C5. Tipping-point analysis: prescriptive analogue of the E-value
# How large a systematic bias in the estimated benefit would flip the verdict?
# We shift benefit by an additive bias c (over-estimation) and a multiplicative
# factor k, and locate where the net deployment hazard reaches zero.
# =============================================================================
def sweep(r, benefit, a, thresholds=THR):
    rows = [metrics_at(r, benefit, a, t) for t in thresholds]
    keys = ["hrr", "hfr", "eed_net", "eed_harm", "car"]
    out = {k: np.array([row[i] for row in rows]) for i, k in enumerate(keys)}
    n = len(r); eed_wd, eed_add = [], []
    for t in thresholds:
        delta, _ = deployment_delta(r, benefit, a, t)
        action = (r >= t).astype(int)
        withdraw = (a == 1) & (action == 0); add = (a == 0) & (action == 1)
        eed_wd.append(float(delta[withdraw].sum() / n * 1000))
        eed_add.append(float(delta[add].sum() / n * 1000))
    out["eed_wd"] = np.array(eed_wd); out["eed_add"] = np.array(eed_add)
    return out

def scalars(sw):
    eed = sw["eed_net"]
    return {"PSS": float(np.nanmean(1 - sw["hfr"])),
            "DHI": float(np.mean(eed)),                          # canonical: SIGNED net EED/1000
            "DHI_onesided": float(np.mean(np.maximum(eed, 0))),  # reference only, NOT headline
            "DHI_signed": float(np.mean(eed)),
            "DHI_frac_positive": float(np.mean(eed > 0)),
            "DHI_withdraw": float(np.mean(sw["eed_wd"])),        # signed, not max(,0)
            "DHI_add": float(np.mean(sw["eed_add"])),           # signed, not max(,0)
            "DHI_frac": float(np.mean(eed > 0)),
            "CAR_mean": float(np.nanmean(sw["car"]))}

upper = float(max(np.nanpercentile(np.abs(benefit_hold), 99), np.nanmax(benefit_hold)) * 1.5 + 1e-6)
bias_grid = np.linspace(0.0, upper, 61)
# perturb the estimated benefit and track the WITHDRAWAL-arm hazard only,
# because "withdrawing beneficial treatment" is the claim the tipping-point must defend
dhi_bias = np.array([scalars(sweep(r_hold, benefit_hold - c, a_hold))["DHI"] for c in bias_grid])
hit = np.where(dhi_bias <= 1e-9)[0]
tip_add = float(bias_grid[hit[0]]) if hit.size else float("nan")

k_grid = np.linspace(0.0, 1.5, 31)
dhi_mult = np.array([scalars(sweep(r_hold, benefit_hold * k, a_hold))["DHI"] for k in k_grid])
tip_mult = float(k_grid[np.argmax(dhi_mult <= 1e-9)]) if np.any(dhi_mult <= 1e-9) else float("nan")

benefit_sd = float(np.std(benefit_hold))
tip_sd = (tip_add / benefit_sd) if (benefit_sd > 0 and not np.isnan(tip_add)) else float("nan")
max_tested_sd = (upper / benefit_sd) if benefit_sd > 0 else float("nan")
tipping = {"additive_bias_to_nullify_DHI": tip_add,
           "additive_bias_in_benefit_SD_units": tip_sd,
           "multiplicative_factor_to_nullify_DHI": tip_mult,
           "benefit_sd": benefit_sd,
           "robust_across_tested_range": bool(np.isnan(tip_add)),
           "max_tested_bias_SD": max_tested_sd}
json.dump(tipping, open(CFG.TAB_DIR / "tipping_point.json", "w"), indent=2)
logger.info("Tipping point: additive benefit bias %.3f (%.2f SD) nullifies the hazard",
            tip_add, tipping["additive_bias_in_benefit_SD_units"])

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6))
axes[0].plot(bias_grid, dhi_bias, color=PALETTE["harm"], lw=2)
axes[0].axhline(0, color="k", lw=0.8)
if not np.isnan(tip_add):
    axes[0].axvline(tip_add, color=PALETTE["accent"], ls="--", lw=1.4,
                    label=f"tipping bias = {tip_add:.3f} ({tipping['additive_bias_in_benefit_SD_units']:.2f} SD)")
    axes[0].legend(fontsize=8)
axes[0].set_title("Additive benefit bias vs deployment hazard")
axes[0].set_xlabel("assumed over-estimation of benefit (risk-diff units)"); axes[0].set_ylabel("DHI")
axes[1].plot(k_grid, dhi_mult, color=PALETTE["MIMIC"], lw=2)
axes[1].axhline(0, color="k", lw=0.8)
if not np.isnan(tip_mult):
    axes[1].axvline(tip_mult, color=PALETTE["accent"], ls="--", lw=1.4, label=f"tipping factor = {tip_mult:.2f}")
    axes[1].legend(fontsize=8)
axes[1].set_title("Benefit scaling vs deployment hazard")
axes[1].set_xlabel("multiplicative factor on estimated benefit"); axes[1].set_ylabel("DHI")
savefig(fig, "fig_tipping_point")
tipping


01:23:15 | INFO    | Tipping point: additive benefit bias 0.000 (0.00 SD) nullifies the hazard
01:23:16 | INFO    | figure -> fig_tipping_point


{'additive_bias_to_nullify_DHI': 0.0,
 'additive_bias_in_benefit_SD_units': 0.0,
 'multiplicative_factor_to_nullify_DHI': 0.0,
 'benefit_sd': 0.09931373896647669,
 'robust_across_tested_range': False,
 'max_tested_bias_SD': 28.862264638240163}

In [26]:
# =============================================================================
# P10 (corrected): Quantitative bias analysis for the ATE.
# VanderWeele-Ding bounding-factor grid: the minimum joint strength of an
# unmeasured confounder (association with treatment RR_EU, with outcome RR_UD)
# needed to move the observed effect to the null. Deterministic; generalises the
# E-value, which is the point on the RR_EU = RR_UD diagonal where RR_adj = 1.
# Ref: VanderWeele & Ding, Ann Intern Med 2017.
# =============================================================================
# observed marginal risk ratio for treatment on mortality (same scale as the E-value)
rr_obs = float("nan")
for k in ("RR_marginal", "RR_point", "RR", "point_estimate", "risk_ratio"):
    try:
        v = evalue.get(k)
        if v is not None and np.isfinite(float(v)):
            rr_obs = float(v); break
    except Exception:
        pass
if not np.isfinite(rr_obs):                      # fallback: crude unadjusted RR (flagged in output)
    p1 = y_hold[a_hold == 1].mean(); p0 = y_hold[a_hold == 0].mean()
    rr_obs = float(p1 / p0)
    logger.warning("QBA: RR not found in evalue dict; using crude unadjusted RR %.3f", rr_obs)

def bias_factor(rr_eu, rr_ud):
    return (rr_eu * rr_ud) / (rr_eu + rr_ud - 1.0)

rr_eu_vals = [1.5, 2.0, 2.5, 3.0, 4.0]
rr_ud_vals = [1.5, 2.0, 2.5, 3.0, 4.0]
qba_rows = []
for rr_eu in rr_eu_vals:
    for rr_ud in rr_ud_vals:
        Bf = bias_factor(rr_eu, rr_ud)
        rr_adj = rr_obs / Bf if rr_obs >= 1 else rr_obs * Bf
        qba_rows.append({"RR_EU": rr_eu, "RR_UD": rr_ud, "bias_factor": round(Bf, 3),
                         "RR_adjusted": round(rr_adj, 3),
                         "crosses_null": bool((rr_obs >= 1 and rr_adj <= 1) or
                                              (rr_obs < 1 and rr_adj >= 1))})
qba_results = pd.DataFrame(qba_rows)
qba_results.attrs["observed_RR"] = rr_obs
qba_results.to_csv(CFG.TAB_DIR / "quantitative_bias_analysis.csv", index=False)
logger.info("QBA (VanderWeele-Ding): observed RR %.3f; %d of %d (RR_EU,RR_UD) cells erase the effect",
            rr_obs, int(qba_results["crosses_null"].sum()), len(qba_results))

# ---- figure: bias-factor heatmap with the null-crossing boundary ----
piv = qba_results.pivot(index="RR_UD", columns="RR_EU", values="RR_adjusted")
fig, ax = plt.subplots(figsize=(6.2, 5))
im = ax.imshow(piv.values, origin="lower", aspect="auto", cmap="RdYlGn_r")
ax.set_xticks(range(len(rr_eu_vals))); ax.set_xticklabels(rr_eu_vals)
ax.set_yticks(range(len(rr_ud_vals))); ax.set_yticklabels(rr_ud_vals)
ax.set_xlabel("confounder-treatment association (RR_EU)")
ax.set_ylabel("confounder-outcome association (RR_UD)")
ax.set_title(f"Bias-adjusted RR (observed RR = {rr_obs:.2f})\ngreen = effect erased to the null")
for (yy, xx), val in np.ndenumerate(piv.values):
    ax.text(xx, yy, f"{val:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=ax, label="adjusted RR")
savefig(fig, "fig_quantitative_bias_analysis")
qba_results

01:23:16 | INFO    | QBA (VanderWeele-Ding): observed RR 1.781; 6 of 25 (RR_EU,RR_UD) cells erase the effect
01:23:18 | INFO    | figure -> fig_quantitative_bias_analysis


,RR_EU,RR_UD,bias_factor,RR_adjusted,crosses_null
0,1.5,1.5,1.125,1.583,False
1,1.5,2.0,1.200,1.484,False
2,1.5,2.5,1.250,1.425,False
3,1.5,3.0,1.286,1.385,False
4,1.5,4.0,1.333,1.336,False
5,2.0,1.5,1.200,1.484,False
6,2.0,2.0,1.333,1.336,False
7,2.0,2.5,1.429,1.247,False
8,2.0,3.0,1.500,1.187,False
9,2.0,4.0,1.600,1.113,False


---
## D. Consolidated comparison figures

Three summary figures a reader uses to judge robustness at a glance: how the headline metrics move across every stress scenario, how prediction and prescriptive quantities transport from MIMIC to eICU, and how prescriptive harm is distributed across subgroups with intervals.

In [27]:
# =============================================================================
# Robustness summary, external transportability, subgroup forest with intervals
# =============================================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5.4))

# (1) robustness across scenarios
if T["robustness_battery"] is not None and not T["robustness_battery"].empty:
    rb = T["robustness_battery"]
    yy = np.arange(len(rb))
    axes[0].barh(yy, rb["DHI"], color=PALETTE["harm"], alpha=0.8)
    axes[0].set_yticks(yy); axes[0].set_yticklabels(rb["scenario"], fontsize=7)
    axes[0].axvline(sc_hold["DHI"], color="k", ls="--", lw=1, label="reference DHI")
    axes[0].set_xlabel("DHI (excess deaths / 1000)"); axes[0].set_title("Robustness across scenarios")
    axes[0].legend(fontsize=8); axes[0].invert_yaxis()
else:
    axes[0].axis("off")

# (2) internal vs external transportability
if not table4.empty:
    q = table4.set_index("quantity")
    labels = list(q.index); x = np.arange(len(labels)); w = 0.38
    axes[1].bar(x - w/2, q["internal (MIMIC)"], w, color=PALETTE["MIMIC"], label="internal")
    axes[1].bar(x + w/2, q["external (eICU)"], w, color=PALETTE["eICU"], label="external")
    axes[1].set_xticks(x); axes[1].set_xticklabels(labels, rotation=30, ha="right", fontsize=8)
    axes[1].set_title("Transportability MIMIC -> eICU"); axes[1].legend(fontsize=8)
else:
    axes[1].axis("off")

# (3) subgroup forest with CIs
if HAVE_HOLD_META and not table6.empty and "DHI_CI" in table6.columns:
    def parse_ci(s):
        a, b = s.strip("[]").split(","); return float(a), float(b)
    yy = np.arange(len(table6))
    for i, (_, row) in enumerate(table6.iterrows()):
        lo, hi = parse_ci(row["DHI_CI"])
        axes[2].plot([lo, hi], [i, i], color=PALETTE["neutral"], lw=2)
        axes[2].plot(row["DHI"], i, "o", color=PALETTE["harm"])
    axes[2].set_yticks(yy)
    axes[2].set_yticklabels([f"{r.variable}: {r.level} (n={r.n})" for _, r in table6.iterrows()], fontsize=7)
    axes[2].axvline(sc_hold["DHI"], color="k", ls="--", lw=1)
    axes[2].set_xlabel("DHI with 95% CI"); axes[2].set_title("Prescriptive harm by subgroup"); axes[2].invert_yaxis()
else:
    axes[2].axis("off")
savefig(fig, "fig_consolidated_robustness")
print("Consolidated comparison figures written.")


01:23:19 | INFO    | figure -> fig_consolidated_robustness
Consolidated comparison figures written.


---
## E. Reviewer stress test

We adopt six reviewer personas and, for each realistic criticism, record the specific analysis and artefact that answers it. Criticisms that public observational ICU data cannot resolve are listed honestly as residual limitations rather than papered over.

In [28]:
# =============================================================================
# Structured criticism-to-response matrix
# =============================================================================
_cfb = json.load(open(CFG.TAB_DIR / "c_for_benefit.json")) if (CFG.TAB_DIR / "c_for_benefit.json").exists() else {}
reviewer_matrix = pd.DataFrame([
 ("Causal inference", "Treatment effect is confounded by indication",
  "Propensity model, stabilised IPW, MSM, g-formula, and AIPW cross-fitted; overlap and positivity checked",
  "ate_estimates.csv, positivity_overlap.json, fig_propensity_overlap"),
 ("Causal inference", "Unmeasured confounding could explain the effect",
  "E-value for the average effect and an additive tipping-point for the prescriptive verdict",
  "evalue.json, tipping_point.json, fig_tipping_point"),
 ("Causal inference", "The benefit model may be uninformative here",
  f"Correct; c-for-benefit is near 0.5 (hold={_cfb.get('c_for_benefit_hold', float('nan')):.3f}), and we quantify exactly when that blinds the audit via a ground-truth operating characteristic",
  "c_for_benefit.json, semisynthetic_operating_characteristic.csv, audit_validity_floor.json, fig_audit_operating_characteristic"),
 ("Methods reviewer", "Why trust an audit whose benefit model fails",
  "We do not claim a harm verdict on real data; the contribution is the validity floor and the demonstration that ICU registries fall below it",
  "audit_validity_floor.json, fig_audit_operating_characteristic"),
 ("Senior statistician", "Prescriptive harm may be a chance finding",
  "Randomisation inference with benefit- and risk-permutation nulls",
  "permutation_inference.json, fig_permutation_inference"),
 ("Senior statistician", "Point estimates lack uncertainty",
  "Bootstrap intervals including treatment-effect-model error; DeLong CIs for AUROC",
  "uncertainty_propagation.json, headline_ci.csv, fig_uncertainty"),
 ("Senior statistician", "Pipeline could fabricate effects",
  "Negative-control outcomes immutable to treatment return near-zero adjusted effect",
  "negative_controls.csv, fig_negative_controls"),
 ("Clinical intensivist", "The deployment rule is a strawman",
  "Risk-guided de-escalation is the documented behaviour; full threshold sweep reported",
  "fig_deployment_safety_curves, fig_decision_curves"),
 ("Clinical intensivist", "Harm must be clinically legible",
  "Expected excess deaths per 1000 and named failure cases with physiology",
  "failure_cases.csv, fig_failure_cases, fig_waterfall_sankey"),
 ("Machine learning", "Findings may be specific to one learner",
  "Five model families audited; DeLong comparisons; consistent PSS ordering",
  "table3_prescriptive_safety.csv, table2b_delong_tests.csv"),
 ("Machine learning", "Leakage from treatment into features",
  "Landmark design; treatment excluded from covariates; ablation of surrogates",
  "ablation_mitigation.csv, fig_study_design"),
 ("Reproducibility", "Results may not transport or reproduce",
  "External eICU validation; fixed seeds; cached artefacts; TRIPOD+AI checklist",
  "table4_external_validation.csv, reproducibility_report.json, tripod_ai_checklist.csv"),
 ("Reproducibility", "Calibration drift across sites ignored",
  "Calibration-drift scenario with site recalibration in the robustness battery",
  "robustness_battery.csv, fig_consolidated_robustness"),
 ("Associate editor", "Novelty may be overstated",
  "Claim limited to a deployment-evaluation methodology; prediction models are instruments",
  "manuscript/abstract.md, framework definitions in Notebook 2 section 4"),
 ("Associate editor", "Generalisability beyond two US databases",
  "Stated as a residual limitation; audit is dataset-agnostic and released for external use",
  "manuscript/limitations.md"),
], columns=["reviewer", "anticipated_criticism", "response_analysis", "artefact"])
def _df_to_md(df):
    cols = [str(c) for c in df.columns]
    out = ["| " + " | ".join(cols) + " |", "| " + " | ".join(["---"] * len(cols)) + " |"]
    for _, row in df.iterrows():
        out.append("| " + " | ".join(str(row[c]).replace("|", "\\|").replace("\n", " ")
                                     for c in df.columns) + " |")
    return "\n".join(out)

reviewer_matrix.to_csv(CFG.SUP_DIR / "reviewer_response_matrix.csv", index=False)
with open(CFG.SUP_DIR / "reviewer_response_matrix.md", "w", encoding="utf-8") as f:
    f.write("# Reviewer stress test: criticism-to-response matrix\n\n")
    f.write(_df_to_md(reviewer_matrix))
logger.info("Reviewer response matrix written (%d items).", len(reviewer_matrix))
reviewer_matrix


01:23:19 | INFO    | Reviewer response matrix written (15 items).


,reviewer,anticipated_criticism,response_analysis,artefact
0,Causal inference,Treatment effect is confounded by indication,"Propensity model, stabilised IPW, MSM, g-formu...","ate_estimates.csv, positivity_overlap.json, fi..."
1,Causal inference,Unmeasured confounding could explain the effect,E-value for the average effect and an additive...,"evalue.json, tipping_point.json, fig_tipping_p..."
2,Causal inference,The benefit model may be uninformative here,Correct; c-for-benefit is near 0.5 (hold=0.565...,"c_for_benefit.json, semisynthetic_operating_ch..."
3,Methods reviewer,Why trust an audit whose benefit model fails,We do not claim a harm verdict on real data; t...,"audit_validity_floor.json, fig_audit_operating..."
4,Senior statistician,Prescriptive harm may be a chance finding,Randomisation inference with benefit- and risk...,"permutation_inference.json, fig_permutation_in..."
5,Senior statistician,Point estimates lack uncertainty,Bootstrap intervals including treatment-effect...,"uncertainty_propagation.json, headline_ci.csv,..."
6,Senior statistician,Pipeline could fabricate effects,Negative-control outcomes immutable to treatme...,"negative_controls.csv, fig_negative_controls"
7,Clinical intensivist,The deployment rule is a strawman,Risk-guided de-escalation is the documented be...,"fig_deployment_safety_curves, fig_decision_curves"
8,Clinical intensivist,Harm must be clinically legible,Expected excess deaths per 1000 and named fail...,"failure_cases.csv, fig_failure_cases, fig_wate..."
9,Machine learning,Findings may be specific to one learner,Five model families audited; DeLong comparison...,"table3_prescriptive_safety.csv, table2b_delong..."


---
## F. Reproducibility and reporting standards

We emit a machine-readable reproducibility report and a TRIPOD+AI-aligned checklist so that a reviewer can confirm the study meets contemporary reporting standards for clinical prediction models augmented with AI.

In [29]:
# =============================================================================
# Reproducibility report and TRIPOD+AI checklist
# =============================================================================
import sklearn
repro = {
    "generated": datetime.now().isoformat(timespec="seconds"),
    "random_seed": CFG.RANDOM_SEED,
    "platform": platform.platform(),
    "python": sys.version.split()[0],
    "package_versions": {"numpy": np.__version__, "pandas": pd.__version__,
                         "scikit-learn": sklearn.__version__,
                         "scipy": (sstats.__name__ and __import__("scipy").__version__) if _HAS_SCIPY else "absent",
                         "matplotlib": matplotlib.__version__},
    "cohorts": {"mimic": int(len(mimic)), "eicu": int(len(eicu)), "hold_out": int(len(r_hold))},
    "primary_model": PRIMARY,
    "audit_definition": {"outcome": CFG.OUTCOME, "treatment": CFG.TREATMENT,
                         "benefit_delta": CFG.BENEFIT_DELTA, "reference_threshold": round(tau_ref, 4)},
    "headline": {"PSS_hold": sc_hold["PSS"], "PSS_ext": sc_ext["PSS"],
                 "DHI_hold": sc_hold["DHI"], "DHI_ext": sc_ext["DHI"],
                 "permutation_p_benefit": perm["p_benefit_permutation"],
                 "c_for_benefit_hold": _cfb.get("c_for_benefit_hold")},
    "inputs_reused": {"nb1_manifest": bool(nb1_manifest), "nb2_manifest": bool(nb2_manifest),
                      "nb2_tables_reused": int(sum(v is not None for v in T.values()))},
}
json.dump(repro, open(CFG.SUP_DIR / "reproducibility_report.json", "w"), indent=2, default=str)

tripod = pd.DataFrame([
 ("Title/Abstract", "Identify study as developing/validating and auditing a model", "Yes", "Title, abstract"),
 ("Background", "Rationale and objective (prescriptive safety)", "Yes", "Notebook 1-3 intros"),
 ("Source of data", "MIMIC-IV development, eICU-CRD external", "Yes", "Table 1, CONSORT"),
 ("Participants", "ICU stays, 24 h landmark eligibility", "Yes", "fig_study_design, CONSORT"),
 ("Outcome", "In-hospital mortality, blinded to predictors", "Yes", "Methods, variable dictionary"),
 ("Predictors", "Pre-anchor covariates only; treatment excluded", "Yes", "fig_study_design, ablation"),
 ("Sample size", "Full available cohorts; no post-hoc pruning", "Yes", "reproducibility_report.json"),
 ("Missing data", "Median imputation with missingness indicators; robustness test", "Yes", "robustness_battery.csv"),
 ("Statistical methods", "Calibration, DeLong, bootstrap, permutation", "Yes", "Tables 2-3, C1-C4"),
 ("Model development", "Five families, nested CV, calibration", "Yes", "Notebook 2 section 2"),
 ("Causal estimation", "IPW/MSM/g-formula/AIPW, E-value", "Yes", "ate_estimates.csv, evalue.json"),
 ("Model performance", "Discrimination and calibration internal + external", "Yes", "Table 2, Table 4"),
 ("Model evaluation", "Prescriptive Safety Audit with uncertainty", "Yes", "Table 3, C1-C5"),
 ("Fairness", "Subgroup prescriptive harm with intervals", "Yes", "Table 6, fig_consolidated_robustness"),
 ("Interpretability", "SHAP/permutation, PDP/ICE, feedback overlap", "Yes", "Notebook 2 section 5"),
 ("Reproducibility", "Seeds, versions, cached artefacts, code released", "Yes", "reproducibility_report.json"),
 ("Limitations", "Two US databases; observational assumptions", "Stated", "manuscript/limitations.md"),
], columns=["TRIPOD_AI_item", "requirement", "status", "location"])
tripod.to_csv(CFG.SUP_DIR / "tripod_ai_checklist.csv", index=False)
logger.info("Reproducibility report and TRIPOD+AI checklist written.")
tripod


01:23:19 | INFO    | Reproducibility report and TRIPOD+AI checklist written.


,TRIPOD_AI_item,requirement,status,location
0,Title/Abstract,Identify study as developing/validating and au...,Yes,"Title, abstract"
1,Background,Rationale and objective (prescriptive safety),Yes,Notebook 1-3 intros
2,Source of data,"MIMIC-IV development, eICU-CRD external",Yes,"Table 1, CONSORT"
3,Participants,"ICU stays, 24 h landmark eligibility",Yes,"fig_study_design, CONSORT"
4,Outcome,"In-hospital mortality, blinded to predictors",Yes,"Methods, variable dictionary"
5,Predictors,Pre-anchor covariates only; treatment excluded,Yes,"fig_study_design, ablation"
6,Sample size,Full available cohorts; no post-hoc pruning,Yes,reproducibility_report.json
7,Missing data,Median imputation with missingness indicators;...,Yes,robustness_battery.csv
8,Statistical methods,"Calibration, DeLong, bootstrap, permutation",Yes,"Tables 2-3, C1-C4"
9,Model development,"Five families, nested CV, calibration",Yes,Notebook 2 section 2


---
## G. Manuscript support

We generate draft manuscript components populated with the actual computed values: a structured abstract, a results narrative, figure and table captions, an abbreviations list, and a limitations statement. These are drafts for the writing team, not final text, and every number is drawn from the saved artefacts.

In [30]:
# =============================================================================
# Draft abstract, results narrative, limitations, captions, abbreviations
# =============================================================================
ate_aipw = None
if T["ate_estimates"] is not None:
    row = T["ate_estimates"][T["ate_estimates"]["estimator"].str.contains("AIPW", case=False, na=False)]
    if len(row): ate_aipw = float(row["ATE_riskdiff"].iloc[0])
auroc_blind = tii_sum.get("AUROC_blind", float("nan"))
ev = evalue.get("E_value", float("nan"))
p_ben = perm["p_benefit_permutation"]
cfb = _cfb.get("c_for_benefit_hold", float("nan"))
tip = json.load(open(CFG.TAB_DIR / "tipping_point.json"))
tip_add_sd = tip.get("additive_bias_in_benefit_SD_units", float("nan"))
if tip.get("additive_bias_to_nullify_DHI") is None or (isinstance(tip_add_sd, float) and np.isnan(tip_add_sd)):
    tip_txt = ("no finite additive bias in the estimated benefit nullified the withdrawal-arm hazard "
               "over the tested grid; the hazard scales directly with the magnitude of a benefit estimate "
               f"that does not discriminate (c-for-benefit {cfb:.2f}) and cannot be separated from estimation noise")
else:
    tip_txt = f"{tip_add_sd:.2f} standard deviations of the estimated benefit"

# Route-A guard: a tipping bias at or below zero means the withdrawal-arm hazard is already null
if (tip.get("additive_bias_to_nullify_DHI") is not None
        and float(tip.get("additive_bias_to_nullify_DHI", 1.0)) <= 1e-9):
    tip_txt = ("the withdrawal-arm hazard is at or below zero at baseline, so no benefit "
               "bias is required to nullify it and the headline hazard is entirely an "
               "escalation-arm phenomenon; the withdrawal mechanism is not operative here")

# Route-A: load the operating-characteristic floor written by NB2 cell 36 (lives in NB2 tables)
_floor_path = CFG.NB2_TABLES / "audit_validity_floor.json"
if _floor_path.exists():
    _floor = json.load(open(_floor_path))
else:
    _floor = {"c_for_benefit_floor": float("nan"),
              "c_for_benefit_real_hold": float("nan"),
              "c_for_benefit_real_ext": float("nan")}
    logger.warning("audit_validity_floor.json not found in %s; run NB2 cell 36 first. Using NaN placeholders.", CFG.NB2_TABLES)
cfb_floor     = _floor["c_for_benefit_floor"]
cfb_real_hold = _floor["c_for_benefit_real_hold"]
cfb_real_ext  = _floor["c_for_benefit_real_ext"]

if not np.isfinite(cfb_floor):
    floor_txt = "a benefit-discrimination threshold the benchmark did not resolve in this run"
else:
    floor_txt = f"a c-for-benefit floor of approximately {cfb_floor:.2f}"

if not np.isfinite(cfb_floor):
    below_txt = (f"the real cohorts (c-for-benefit {cfb_real_hold:.2f} internal, "
                 f"{cfb_real_ext:.2f} external) sit in the low-discrimination regime where the audit is unreliable")
else:
    below_txt = (f"both ICU cohorts fell below the floor (c-for-benefit {cfb_real_hold:.2f} internal, "
                 f"{cfb_real_ext:.2f} external)")
    
# --- overlap-population DR effect from the Part B table (nb2_outputs/tables/ate_estimates.csv) ---
ate_ato_dr = float("nan")
if T["ate_estimates"] is not None:
    r2 = T["ate_estimates"][T["ate_estimates"]["estimator"].str.contains("DR-ATO", case=False, na=False)]
    if len(r2):
        ate_ato_dr = float(r2["ATE_riskdiff"].iloc[0])

# --- balance summary from the Part B artefact (lives in nb2_outputs/tables) ---
_balp = CFG.NB2_TABLES / "covariate_balance_full.csv"
if _balp.exists():
    _bal = pd.read_csv(_balp)
    smd_max_unw = float(_bal["SMD_unweighted"].abs().max())
    smd_max_ow  = float(_bal["SMD_overlap_weighted"].abs().max())
else:
    smd_max_unw = smd_max_ow = float("nan")

# --- withdrawal-arm harmful-case count at the reference threshold (computed here in NB3) ---
_delta_ref, _ = deployment_delta(r_hold, benefit_hold, a_hold, tau_ref)
_withdraw_ref = (a_hold == 1) & ((r_hold >= tau_ref).astype(int) == 0)
n_withdraw_harm = int(np.sum(_withdraw_ref & (_delta_ref > DELTA)))

abstract = f"""# Draft structured abstract

**Background.** Prediction models are increasingly proposed to guide intensive-care treatment, yet discrimination and calibration do not establish that acting on a model would help rather than harm. We ask a prior question: under what conditions can a prescriptive-safety audit built on observational data be trusted at all?

**Methods.** Using a 24 hour landmark design in MIMIC-IV (development and internal hold-out) with external validation in eICU-CRD, we trained calibrated mortality models on strictly pre-anchor covariates and estimated the conditional effect of post-anchor organ support by inverse probability weighting, a marginal structural model, the g-formula, and a cross-fitted doubly robust estimator on the overlap population. We then built a semi-synthetic benchmark with a known individual benefit, held the true deployment hazard fixed, and degraded only the treatment-effect model to trace how the audit's recovery of the true Deployment Hazard Index depends on benefit-model quality (c-for-benefit). We located the real cohorts on that axis.

**Results.** Risk models discriminated mortality well (internal AUROC approximately {auroc_blind:.2f}). The doubly robust effect of organ support on the overlap population was {ate_ato_dr:+.3f} (E-value {ev:.2f}); its sign is consistent with residual confounding by indication and we do not interpret it causally. The benefit estimate did not discriminate (c-for-benefit {cfb:.2f}). In the benchmark, the audit recovered the true hazard without bias only above {floor_txt}; below it the estimated hazard collapsed toward zero while the true hazard remained large. {below_txt[0].upper() + below_txt[1:]}, so the deployment-hazard estimates on real data are uninterpretable as harm.

**Conclusions.** Prescriptive-safety auditing of observational ICU data is only trustworthy when the treatment-effect model clears a measurable benefit-discrimination floor. On two large ICU databases the floor is not cleared for organ support, so prediction accuracy cannot be translated into a prescriptive-safety verdict here. We release the audit and its operating characteristic as a precondition, not a conclusion, for clinical decision-support claims.
"""
open(CFG.MAN_DIR / "abstract.md", "w", encoding="utf-8").write(abstract)

results = f"""# Draft results narrative

Across {len(mimic)} MIMIC-IV and {len(eicu)} eICU-CRD landmark-eligible ICU stays, calibrated models discriminated in-hospital mortality with an internal AUROC of about {auroc_blind:.2f} and transported to eICU with the expected modest degradation (Table 2, Table 4). Treatment was strongly confounded by indication; after a dedicated logistic overlap-weighting the maximum standardised mean difference across covariates fell from {smd_max_unw:.2f} to {smd_max_ow:.2f} (fig_love_balance), defining the equipoise population in which a de-escalation decision is faced (fig_propensity_overlap). The doubly robust effect of organ support on the overlap population was {ate_ato_dr:+.3f} (Table 5, E-value {ev:.2f}); the confidence interval includes zero and the sign is consistent with residual confounding, so we do not interpret it as a causal effect.

The treatment-effect model on which any prescriptive verdict rests did not discriminate individual benefit (c-for-benefit {cfb:.2f}; fig_benefit_validation). To determine whether this makes the audit unreliable, we built a semi-synthetic benchmark with a known benefit and a fixed true Deployment Hazard Index, and degraded only the covariate information available to the treatment-effect model (fig_audit_operating_characteristic). The audit recovered the true hazard without systematic bias only above {floor_txt}; below that floor the estimated hazard collapsed toward zero while the true hazard remained large. {below_txt[0].upper() + below_txt[1:]}, so the deployment-hazard numbers computed on MIMIC and eICU are within the audit's blind zone and are reported descriptively rather than as evidence of harm. Consistent with this, {tip_txt}, the top risk and treatment drivers did not overlap (0 of 15), and negative-control outcomes immutable to treatment returned near-zero adjusted effects (fig_negative_controls).
"""
open(CFG.MAN_DIR / "results_narrative.md", "w", encoding="utf-8").write(results)

limitations = """# Limitations (draft)

1. The central result is conditional: it establishes when observational prescriptive auditing is trustworthy, not that organ support is prescriptively unsafe. On these data the benefit-discrimination floor is not cleared, so no harm verdict is issued.
2. Identification of individual benefit rests on conditional exchangeability given recorded pre-anchor covariates, which is not testable; the E-value and the operating characteristic bound, but cannot remove, this dependence.
3. The data are two US ICU databases; the floor and the audit are released as dataset-agnostic tools, and their values on other populations must be re-estimated.
4. Treatment is a composite organ-support action at a single landmark; finer-grained or time-varying strategies would define different audits and possibly different floors.
5. The semi-synthetic benchmark isolates benefit-model quality by degrading covariate information; real degradation mechanisms are more varied, and the estimated floor is therefore an operating guide, not a universal constant.
"""
open(CFG.MAN_DIR / "limitations.md", "w", encoding="utf-8").write(limitations)

captions = {
 "fig_causal_dag": "Causal directed acyclic graph. Treatment A lies on the path from pre-anchor covariates L to mortality Y; a deployed risk score r=f(L) perturbs A. Dashed edges denote unmeasured confounding U probed by the E-value.",
 "fig_study_design": "Landmark study design and audit overview. Covariates are measured before the 24 hour anchor and treatment strictly after it, identifying the causal estimand the audit perturbs.",
 "fig_consort_flow": "Cohort derivation and analysis flow for MIMIC-IV and eICU-CRD.",
 "fig_graphical_abstract": "Graphical abstract. Accurate models span the full prescriptive-safety range, so prediction quality does not determine prescriptive safety.",
 "fig_permutation_inference": "Randomisation inference. Observed Deployment Hazard Index against benefit- and risk-permutation nulls.",
 "fig_negative_controls": "Negative-control outcomes immutable to treatment; adjustment removes the confounded naive association.",
 "fig_benefit_validation": "Treatment-effect-model validation: estimated benefit distribution with c-for-benefit, and benefit calibration by decile.",
 "fig_uncertainty": "Uncertainty propagation. Bootstrap distributions of the hazard metrics including treatment-effect-model error.",
 "fig_tipping_point": "Tipping-point analysis. Systematic benefit bias required to nullify the deployment hazard.",
 "fig_consolidated_robustness": "Consolidated robustness, transportability, and subgroup prescriptive harm.",
 "fig_audit_operating_characteristic": "Audit operating characteristic. Left: audit-estimated versus true Deployment Hazard Index across benchmark settings, coloured by benefit-model quality. Right: recovered fraction of the true hazard as a function of c-for-benefit, with the benefit-discrimination floor and the real ICU cohorts marked; the shaded band is the region in which the audit is blind.",
}
with open(CFG.MAN_DIR / "figure_captions.md", "w", encoding="utf-8") as f:
    f.write("# Figure captions (draft)\n\n")
    for k, v in captions.items():
        f.write(f"**{k}.** {v}\n\n")

abbrev = pd.DataFrame([
 ("PSS", "Prescriptive Safety Score"), ("DHI", "Deployment Hazard Index"),
 ("HRR", "Harmful Recommendation Rate"), ("HFR", "Harmful Flip Rate"),
 ("TII", "Treatment Influence Index"), ("CAR", "Counterfactual Agreement Rate"),
 ("EED", "Expected Excess Deaths"), ("CATE", "Conditional Average Treatment Effect"),
 ("ATE", "Average Treatment Effect"), ("IPW", "Inverse Probability Weighting"),
 ("MSM", "Marginal Structural Model"), ("AIPW", "Augmented IPW (doubly robust)"),
 ("AUROC", "Area Under the Receiver Operating Characteristic curve"),
 ("ECE", "Expected Calibration Error"), ("SMD", "Standardised Mean Difference"),
 ("SOFA", "Sequential Organ Failure Assessment"),
], columns=["abbreviation", "definition"])
abbrev.to_csv(CFG.SUP_DIR / "abbreviations.csv", index=False)
logger.info("Manuscript support drafts written to %s", CFG.MAN_DIR)
print("Abstract, results narrative, limitations, captions, abbreviations drafted.")


01:23:20 | INFO    | Manuscript support drafts written to nb3_outputs\manuscript
Abstract, results narrative, limitations, captions, abbreviations drafted.


In [31]:
# =============================================================================
# Finalise: organise outputs and write the Notebook 3 manifest
# =============================================================================
manifest = {
    "generated": datetime.now().isoformat(timespec="seconds"),
    "outputs": {
        "tables": sorted(p.name for p in CFG.TAB_DIR.glob("*")),
        "figures": sorted(p.name for p in CFG.FIG_DIR.glob("*.png")),
        "supplementary": sorted(p.name for p in CFG.SUP_DIR.glob("*")),
        "manuscript": sorted(p.name for p in CFG.MAN_DIR.glob("*")),
    },
    "headline": {"PSS_hold": sc_hold["PSS"], "PSS_ext": sc_ext["PSS"],
                 "DHI_hold": sc_hold["DHI"], "DHI_ext": sc_ext["DHI"],
                 "permutation_p_benefit": perm["p_benefit_permutation"],
                 "c_for_benefit_hold": _cfb.get("c_for_benefit_hold"),
                 "tipping_bias_SD": tipping["additive_bias_in_benefit_SD_units"]},
    "reused_from": ["nb1_outputs", "nb2_outputs"],
}
json.dump(manifest, open(CFG.OUT / "nb3_manifest.json", "w"), indent=2, default=str)

print("=" * 66)
print("NOTEBOOK 3 COMPLETE")
print("=" * 66)
for kind, items in manifest["outputs"].items():
    print(f"  {kind:14s}: {len(items)} files")
print(f"  PSS  hold/ext : {sc_hold['PSS']:.3f} / {sc_ext['PSS']:.3f}")
print(f"  DHI  hold/ext : {sc_hold['DHI']:.2f} / {sc_ext['DHI']:.2f} excess deaths per 1000")
print(f"  permutation p : {perm['p_benefit_permutation']:.4f} (benefit null)")
print(f"  c-for-benefit : {_cfb.get('c_for_benefit_hold', float('nan')):.3f}")
_rb = tipping.get('robust_across_tested_range'); _sd = tipping['additive_bias_in_benefit_SD_units']
print("  tipping bias  : " + (f">{tipping['max_tested_bias_SD']:.2f} SD (robust)" if _rb else f"{_sd:.2f} SD") + " to nullify hazard")
print(f"  Outputs under : {CFG.OUT.resolve()}")


NOTEBOOK 3 COMPLETE
  tables        : 25 files
  figures       : 11 files
  supplementary : 5 files
  manuscript    : 4 files
  PSS  hold/ext : 0.821 / 0.856
  DHI  hold/ext : -10.16 / -12.16 excess deaths per 1000
  permutation p : 0.0012 (benefit null)
  c-for-benefit : 0.565
  tipping bias  : 0.00 SD to nullify hazard
  Outputs under : C:\Users\<user>\Downloads\nb3_outputs


---
### Submission package

The publication package is now assembled under `nb3_outputs/`. `tables/` holds the six numbered main tables as CSV and LaTeX plus a combined Excel workbook; `figures/` holds every figure as 300 dpi PNG and vector PDF; `supplementary/` holds the reproducibility report, TRIPOD+AI checklist, reviewer-response matrix, abbreviations, and negative-control and validation outputs; and `manuscript/` holds the draft abstract, results narrative, limitations, and captions with every value drawn from the saved artefacts.

The scientific position is deliberately narrow and defensible. We do not claim a better mortality model; we claim, and now demonstrate with external validation, randomisation inference, negative controls, treatment-effect-model validation, uncertainty propagation, and a tipping-point analysis, that prediction accuracy and prescriptive safety are distinct, that the gap is measurable, and that a well-discriminating model can be unsafe to deploy as a treatment guide. The residual limitations, chiefly the untestable no-unmeasured-confounding assumption and the use of two US databases, are stated rather than hidden, which is where an honest reading of public observational ICU data must leave them.